Resource
-

In [1]:
!pip install wandb==0.22.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 46.4 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.1
    Uninstalling wandb-0.25.1:
      Successfully uninstalled wandb-0.25.1


### Install kaleido for export pdf

In [2]:
!pip install plotly==6.4.0
!pip install kaleido==1.2.0
!pip install choreographer==1.2.1
!pip install pypdf
!apt update
!apt-get install libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 25.0 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: choreographer
    Found existing installation: choreographer 1.4.0
    Uninstalling choreographer-1.4.0:
      Successfully uninstalled choreographer-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.7 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security 

In [3]:
import plotly.io as pio

pio.get_chrome()

PosixPath('/usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

# Login

In [1]:
from google.colab import userdata
WANDB_KEY = userdata.get('WANDB_TOKEN')

In [2]:
import wandb
print(wandb.__version__)
wandb.login(key=WANDB_KEY)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

0.22.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rofifpriyo (rofifpriyo-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Parameter

In [3]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

ENTITY = wandb.apis.PublicApi().default_entity
PROJECT = "calibration-on-quantized-multilingual"

# Plot from WandB

> Docs: https://docs.wandb.ai/ref/python/public-api/api/#method-apiruns

## Perplexity

### Deliverables + Functions

#### 3x Run Stabilizer

In [4]:
import numpy as np
import re
def perplexity_heatmap_deliver_vertical(eval_languages, runs, base_models, quantization_technique, bit_width, eval_dataset, custom_title="", display_mean=True, compile=False, show_subline=True, hf_evaluate=False, zscore_threshold=1, gmean_aggregate=False, two_color=True, display_colorbar=True, write_pdf=""):
  language_sort = {
      # # Flores Plus
      # "eng_Latn": 0, "fra_Latn": 1, "hin_Deva": 2, "npi_Deva": 3, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "cmn_Hans": 12, "jpn_Jpan": 13, "kor_Hang": 14, "arb_Arab" : 15,\
      # Wikipedia
      "en": 0, "fr": 1, "hi": 2, "ne": 3, "ta": 6, "te": 7, "id": 8, "ms": 9, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  } # Add floresplus evals, and count their respective language's family
  family_counter = []
  if eval_dataset == 'floresplus':
    # flores_plus_add = ["afr_Latn", "dan_Latn", "deu_Latn", "eng_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"]
    flores_plus_add = ["eng_Latn", "tpi_Latn", "afr_Latn", "deu_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "ydd_Hebr", "dan_Latn", "fao_Latn", "isl_Latn", "nno_Latn", "nob_Latn", "swe_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 0 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["kan_Knda", "mal_Mlym", "tam_Taml", "tel_Telu"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 6 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    # flores_plus_add = ["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "ind_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"]
    flores_plus_add = ["ind_Latn", "zsm_Latn", "bjn_Latn", "min_Latn", "ace_Latn", "ban_Latn", "bug_Latn", "jav_Latn", "sun_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "fil_Latn", "war_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 8 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "swh_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 10 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant", "cmn_Hans"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 12 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
  elif eval_dataset == 'wikipedia':
    wikipedia_add = ["en", "de", "nl"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 0 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["ml", "ta", "te"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 6 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["id", "tl", "ms"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 8 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["sw", "lg", "yo"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 10 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["bo", "my", "zh"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 12 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
  eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
  # print(eval_languages)

  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language"]
  columns.extend(eval_languages)
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "CohereLabs/aya-expanse-8b": 1, "meta-llama/Llama-3.1-8B-Instruct": 2}

  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "CohereLabs/aya-expanse-8b": "Aya", "meta-llama/Llama-3.1-8B-Instruct": "Llama"}
  quantization_technique_label = {"UQ": "Unquantized", "gptq": "GPTQ", "slimllm": "SliM-LLM", "tacq": "TaCQ"}
  eval_label = {
      # # Flores Plus
      "eng_Latn": "EN", "fra_Latn": "FR", "hin_Deva": "HI", "npi_Deva": "NE", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "zsm_Latn": "ZSM", "swh_Latn": "SW", "yor_Latn": "YO", "cmn_Hans": "ZH", "jpn_Jpan": "JA", "kor_Hang": "KO", "arb_Arab" : "AR",\
      "ace_Latn": "ACE", "ban_Latn": "BAN", "bjn_Latn": "BJN", "bug_Latn": "BUG", "ceb_Latn": "CEB", "fij_Latn": "FIJ", "ilo_Latn": "ILO", "jav_Latn": "JAV", "min_Latn": "MIN", "plt_Latn": "PLT", "mri_Latn": "MRI", "pag_Latn": "PAG", "smo_Latn": "SMO", "sun_Latn": "SUN", "fil_Latn": "FIL", "war_Latn": "WAR",\
      # Wikipedia
      "en": "EN", "fr": "FR", "hi": "HI", "ne": "NE", "ta": "TA", "te": "TE", "id": "ID", "ms": "MS", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # # Belebele
      # "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TE", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
  }

  data_acc = []
  data_nancount = []

  data_gptq = []
  data_slimllm = []
  data_tacq = []

  # import numpy as np
  # data_slimllm.append(["-", "CohereLabs/aya-expanse-8b", "slimllm", "4-bit", "Tamil"] + [np.nan for _ in eval_languages])

  unquantized_data_acc = []

  # Iterate over the runs and print details
  for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue

      run_data_acc = []
      run_data_nancount = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      run_data_nancount.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      if not hf_evaluate:
        for lang in eval_languages:
          if lang in ["zh", "Chinese", "cmn_Hans", "ja", "Japanese", "jpn_Jpan", "bo", "Tibetan", "bod_Tibt", "dz", "Dzongkha", "dzo_Tibt", "Yue Chinese", "yue_Hant"]:
            acc = run.summary.get(f"{eval_dataset.lower()}_byte_acc_{lang}", None)
            nancount = run.summary.get(f"{eval_dataset.lower()}_nan_count_{lang}", 0)
            if acc:
              # acc = math.pow(acc, 1/3)
              pass
            run_data_acc.append(acc)
            run_data_nancount.append(nancount)
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_token_perplexity_{lang}",
                                                run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                                ))
            run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nan_count_{lang}", 0))
      elif hf_evaluate:
        for lang in eval_languages:
          if gmean_aggregate:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_gmean_perplexity_{lang}",
                                                run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                                ))
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_token_perplexity_{lang}",
                                                run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                                ))
          run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nancount_{lang}", 0))
      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "gptq":
        data_gptq.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "slimllm":
        data_slimllm.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "tacq":
        data_tacq.append(run_data_acc)
        # data_nancount.append(run_data_nancount)

  df_unquantized_acc_chief = pd.DataFrame(unquantized_data_acc, columns=columns)
  df_gptq_chief = pd.DataFrame(data_gptq, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_slimllm_chief = pd.DataFrame(data_slimllm, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_tacq_chief = pd.DataFrame(data_tacq, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  display(pd.concat([df_unquantized_acc_chief, df_gptq_chief, df_slimllm_chief, df_tacq_chief], axis=0))

  # with pd.option_context('display.max_rows', None):

  # Unquantized
  df_unquantized_acc_chief_std = df_unquantized_acc_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std")

  # GPTQ agg
  df_gptq_chief_std = df_gptq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_gptq_chief = df_gptq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  # Slim agg
  df_slimllm_chief_std = df_slimllm_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_slimllm_chief = df_slimllm_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  # Tacq agg
  df_tacq_chief_std = df_tacq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_tacq_chief = df_tacq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  with pd.option_context('display.max_rows', None):
    display(df_tacq_chief_std)
  display(df_gptq_chief.columns)

  # display(df_gptq_chief_acc)
  # display(df_gptq_chief_std)

  df_gptq_chief[df_gptq_chief.columns[4:]] = df_gptq_chief[df_gptq_chief.columns[4:]].astype(float)
  df_slimllm_chief[df_slimllm_chief.columns[4:]] = df_slimllm_chief[df_slimllm_chief.columns[4:]].astype(float)
  df_tacq_chief[df_tacq_chief.columns[4:]] = df_tacq_chief[df_tacq_chief.columns[4:]].astype(float)
  df_unquantized_acc_chief[df_unquantized_acc_chief.columns[4:]] = df_unquantized_acc_chief[df_unquantized_acc_chief.columns[4:]].astype(float)

  # Easying aggregate calculation
  # Perplexity related, change order of PPL
  df_gptq_chief.iloc[:, 4:] = df_gptq_chief.iloc[:, 4:].apply(np.log10)
  df_slimllm_chief.iloc[:, 4:] = df_slimllm_chief.iloc[:, 4:].apply(np.log10)
  df_tacq_chief.iloc[:, 4:] = df_tacq_chief.iloc[:, 4:].apply(np.log10)
  df_unquantized_acc_chief.iloc[:, 4:] = df_unquantized_acc_chief.iloc[:, 4:].apply(np.log10)


  fig = go.Figure()

  df_chief = pd.DataFrame([])
  df_chief_std = pd.DataFrame([])
  df_zscore_chief = pd.DataFrame([])
  index_border_chief = [(0, 0)]
  # for base_model in ["CohereLabs/aya-expanse-8b"]:
  for base_model in base_models:
    print(f"start model: {base_model}")
    df_gptq = df_gptq_chief.loc[df_gptq_chief["Base Model"] == base_model]
    df_slimllm = df_slimllm_chief.loc[df_slimllm_chief["Base Model"] == base_model]
    df_tacq = df_tacq_chief.loc[df_tacq_chief["Base Model"] == base_model]
    df_unquantized_acc = df_unquantized_acc_chief.loc[df_unquantized_acc_chief["Base Model"] == base_model]
    df_gptq_std = df_gptq_chief_std.loc[df_gptq_chief_std["Base Model"] == base_model]
    df_slimllm_std = df_slimllm_chief_std.loc[df_slimllm_chief_std["Base Model"] == base_model]
    df_tacq_std = df_tacq_chief_std.loc[df_tacq_chief_std["Base Model"] == base_model]
    df_unquantized_acc_std = df_unquantized_acc_chief_std.loc[df_unquantized_acc_chief_std["Base Model"] == base_model]
    display(df_gptq)

    # display(df_unquantized_acc)
    base_model = base_model_label[base_model]

    df_gptq_std = df_gptq_std.iloc[:, 4:].T.rename(df_gptq_std['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_gptq_std['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_slimllm_std = df_slimllm_std.iloc[:, 4:].T.rename(df_slimllm_std['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_slimllm_std['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_tacq_std = df_tacq_std.iloc[:, 4:].T.rename(df_tacq_std['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_tacq_std['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_unquantized_acc_std = df_unquantized_acc_std.iloc[:, 4:].T.rename(lambda x: "UQ", axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')

    # Take only the result and reformat the indices
    # df_gptq = df_gptq.iloc[:, 5:].sub(df_unquantized_acc.iloc[:, 5:].values).T.rename(df_gptq['Quantization Technique'].apply(lambda x: x[0].upper()) + '-' + df_gptq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_gptq = df_gptq.iloc[:, 4:].T.rename(df_gptq['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_gptq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_gptq = pd.concat([df_gptq.T, df_gptq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_gptq = df_gptq.astype(float)
    display(df_gptq)

    # df_slimllm = df_slimllm.iloc[:, 5:].sub(df_unquantized_acc.iloc[:, 5:].values).T.rename(df_slimllm['Quantization Technique'].apply(lambda x: x[0].upper()) + '-' + df_slimllm['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_slimllm = df_slimllm.iloc[:, 4:].T.rename(df_slimllm['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_slimllm['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_slimllm = pd.concat([df_slimllm.T, df_slimllm.mean(axis=0).to_frame('Mean')], axis=1).T
    df_slimllm = df_slimllm.astype(float)

    # df_tacq = df_tacq.iloc[:, 5:].sub(df_unquantized_acc.iloc[:, 5:].values).T.rename(df_tacq['Quantization Technique'].apply(lambda x: x[0].upper()) + '-' + df_tacq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_tacq = df_tacq.iloc[:, 4:].T.rename(df_tacq['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_tacq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_tacq = pd.concat([df_tacq.T, df_tacq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_tacq = df_tacq.astype(float)

    df_unquantized_acc = df_unquantized_acc.iloc[:, 4:].T.rename(lambda x: "UQ", axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_unquantized_acc = pd.concat([df_unquantized_acc, df_unquantized_acc.mean(axis=0).to_frame(f'Mean_{base_model}').T], axis=0)
    df = pd.concat([df_gptq, df_slimllm, df_tacq], axis=1)
    df_std = pd.concat([df_gptq_std, df_slimllm_std, df_tacq_std], axis=1)
    df_mean_quantized = df.mean(axis=1).to_frame('Mean of Quantized')
    # display(df_unquantized_acc)

    for iter, (index_row, index_col) in enumerate([df_unquantized_acc.shape, df_gptq.shape, df_slimllm.shape, df_tacq.shape]):
      if iter != 0:
        index_row = 0
      index_border_chief.append((index_border_chief[-1][0] + index_row, index_border_chief[-1][1] + index_col))

    from scipy.stats import zscore
    df_zscore_gptq = pd.DataFrame(zscore(df_gptq, axis=1, nan_policy='omit'), columns=df_gptq.columns, index=df_gptq.index)
    df_zscore_slimllm = pd.DataFrame(zscore(df_slimllm, axis=1, nan_policy='omit'), columns=df_slimllm.columns, index=df_slimllm.index)
    df_zscore_tacq = pd.DataFrame(zscore(df_tacq, axis=1, nan_policy='omit'), columns=df_tacq.columns, index=df_tacq.index)

    # Skip mean of quantized
    df_zscore = pd.concat([pd.Series(np.zeros_like(df_zscore_tacq.index), index=df_zscore_tacq.index, name="UQ"), df_zscore_gptq, df_zscore_slimllm, df_zscore_tacq], axis=1)

    df = pd.concat([df_unquantized_acc, df], axis=1)
    df_chief = pd.concat([df_chief, df], axis=0)
    df_zscore_chief = pd.concat([df_zscore_chief, df_zscore], axis=0)

    df_std = pd.concat([df_unquantized_acc_std, df_std], axis=1)
    df_chief_std = pd.concat([df_chief_std, df_std], axis=0)

  # Revariable for onness usage of df, we was using chief for the sake of differentiating between chief and iter df
  df = df_chief
  df_std = df_chief_std
  df_zscore = df_zscore_chief
  df = df.rename(lambda x: eval_label.get(x, x), axis=0)
  df_std = df_std.rename(lambda x: eval_label.get(x, x), axis=0)
  df_zscore = df_zscore.rename(lambda x: eval_label.get(x, x), axis=0)
  # display(df)
  # display(df_zscore)

  # Rebase the number
  df = df.apply(lambda x: 10 ** x)

  # display(df)
  # display(df_zscore)
  if not display_mean:
    df = df.reindex(index=df.index[::-1])
    df_std = df_std.reindex(index=df_std.index[::-1])
    df_zscore = df_zscore.reindex(index=df_zscore.index[::-1])
  display(df)
  display(df_std)

  # df_text = df.copy().map(lambda x: f"x10^{x:.1f}" if not np.isnan(x) else "-")
  # df_text = df.copy().map(lambda x: f"{(10**x):.2f}" if x < 3 else f"{(10**x):.1e}" if not np.isnan(x) else "-")
  df_std = df_std.map(lambda x: f"{x:.2f}" if 0 < x < 100 else f"{x:.1f}" if 0 < x < 1000 else re.sub(r'([eE][+-])0+(?=\d)', r'\1', f"{x:.1e}") if (not np.isnan(x) and not x == 0) else "")
  df_text = df.copy().map(lambda x: f"{x:.2f}" if x < 100 else f"{x:.1f}" if x < 1000 else re.sub(r'([eE][+-])0+(?=\d)', r'\1', f"{x:.1e}") if not np.isnan(x) else "-")
  # df_text = list(zip(df_text.values, df_std.values))
  print("## Shape")
  print(f"df text: {df_text.shape}")
  print(f"df std: {df_std.shape}")
  df_text = [list(zip(x, y)) for x, y in list(zip(df_text.values.tolist(), df_std.values.tolist()))]
  # df_text.iloc[:, 0] = df_text.iloc[:, 0].apply(lambda x: x[1:])

  # Color palette use this: Green: https://colordesigner.io/gradient-generator from 255, 255, 255 to 6, 54, 21, Red: https://colordesigner.io/gradient-generator from 255, 255, 255 to 214, 39, 40
  def exact_zscore_scale(z, df_zscore):
    return (z - df_zscore.min(axis=None)) / (df_zscore.max(axis=None) - df_zscore.min(axis=None))
  # Add 0, then iteratively add green and red until it touch min 0 or max 1
  colorscale = []
  # If light = good, dark = bad
  if not two_color:
    colorscale = [[0, 'rgba(255, 255, 255, 0.85)'], [1, 'rgba(214, 39, 40, 0.85)']]
  # For two color signaling, e.g. red = bad, green = good
  else:
    colorscale = [[exact_zscore_scale(0, df_zscore), 'rgba(255, 255, 255, 0.85)']]
    # Green for negativee z   1. Three scale different color, 2. Normal, 3. Thresholded z-score
    if zscore_threshold >= 1:
      color_zip = zip([-1+10**-5, -1, -2, -3], ['rgba(255, 255, 255, 0.85)', 'rgba(144, 166, 151, 0.85)', 'rgba(61, 99, 73, 0.85)', 'rgba(6,54,21, 0.85)'])
    elif zscore_threshold >= 1/2:
      color_zip = zip([-1/2+10**-5, -1/2, -2, -3], ['rgba(255, 255, 255, 0.85)', 'rgba(144, 166, 151, 0.85)', 'rgba(61, 99, 73, 0.85)', 'rgba(6,54,21, 0.85)'])
    else:
      # color_zip = zip([-1, -2, -3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
      color_zip = zip([-3], ['rgba(6,54,21, 0.85)'])
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    for z, color in color_zip:
    # for z, color in zip([-10**-5, -1, -1-10**-5, -2, -3], ['rgba(0,255,255, 0.85)', 'rgba(0,255,255, 0.85)', 'rgba(6,54,21, 0.85)', 'rgba(6,54,21, 0.85)', 'rgba(191,139,255, 0.85)']):
    # for z, color in zip([-1, -1-10**(-5), -2], ['rgba(144, 166, 151, 0.85)', 'rgba(89, 121, 99, 0.85)', 'rgba(6,54,21, 0.85)']):
    # for z, color in zip([-1+10**-5, -1, -2, -3], ['rgba(255, 255, 255, 0.85)', 'rgba(144, 166, 151, 0.85)', 'rgba(61, 99, 73, 0.85)', 'rgba(6,54,21, 0.85)']):
      z_scale = exact_zscore_scale(z, df_zscore)
      colorscale.insert(0, [max(0, z_scale), color])
      if z_scale < 0:
        break
    # Red for positive z
    if zscore_threshold >= 1:
      color_zip = zip([1-10**-5, 1, 2, 3], ['rgba(255, 255, 255, 0.85)', 'rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
    elif zscore_threshold >= 1/2:
      color_zip = zip([1/2-10**-5, 1/2, 2, 3], ['rgba(255, 255, 255, 0.85)', 'rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
    else:
      # color_zip = zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
      color_zip = zip([3], ['rgba(214, 39, 40, 0.85)'])
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    for z, color in color_zip:
      z_scale = exact_zscore_scale(z, df_zscore)
      colorscale.append([min(1, z_scale), color])
      if z_scale > 1:
        break
  print("Display df")
  display(df)
  print("Display df std")
  display(df_std)
  fig = go.Figure(data=go.Heatmap(
                      z=df_zscore,
                      x=[[col_name.split("-")[0] if col_name != "UQ" else "16-bit" for col_name in df_zscore.columns], [col_name.split("-")[-1] if col_name != "UQ" else "-" for col_name in df_zscore.columns]],
                      y=[[f"""<span style='rotate: 0.5turn; display: inline-block;'> {idx_name.split("_")[-1]} </span>""" for idx_name in df_zscore.index],
                       [f"""<span style='rotate: -90deg;'> {idx_name.split("_")[-0]} </span>""" for idx_name in df_zscore.index]],
                      # text=[list(zip(x, y)) for x, y in list(zip(pd.concat([df_unquantized_acc, df, df_mean_quantized], axis=0).T.values.tolist(), df_nancount_all.T.values.tolist()))],
                      text=df_text,
                      # texttemplate="×10^%{text:.1f}",
                      texttemplate="%{text[0]}<sub>%{text[1]}</sub>",
                      # colorscale='Viridis',
                      # colorscale= [[0, 'rgba(6,54,21, 0.85)'],
                      #               [(0 - df_zscore.min(axis=None)) / (df_zscore.max(axis=None) - df_zscore.min(axis=None)), 'rgba(255, 255, 255, 0.85)'],
                      #               [1, 'rgba(214, 39, 40, 0.85)']],
                      colorscale=colorscale,
                      # colorbar=dict(title='Z-score'),
                      textfont={"size":16, "family": "Arial Black"},
                  ))
  # print(colorscale)

  # Horizontal each Eval Languages
  row_gptq, col_gptq = df_gptq.shape
  row_slimllm, col_slimllm = df_slimllm.shape
  row_tacq, col_tacq = df_tacq.shape
  row, col = df_zscore.shape
  offset_mean = 0 if display_mean else 1
  # Horizontal Line
  fig.add_shape(type="line",
                x0=-0.5, y0=row+offset_mean-1.5, x1=col-0.5, y1=row+offset_mean-1.5,
                line=dict(color="black", width = 1),
                )
  # # Vertical Line
  # fig.add_shape(type="line",
  #               x0=col-1.5, y0=-0.5, x1=col-1.5, y1=row-1.5,
  #               line=dict(color="black", width = 5),
  #               )
  # Vertical Each Quant Tech
  # print(index_border_chief)
  for (row_each, col_each) in index_border_chief[:5]:
    fig.add_shape(type="line",
                  x0=col_each-0.5, y0=-0.5, x1=col_each-0.5, y1=row+offset_mean-1.5,
                  line=dict(color="black", width = 5),
                  )
  # Horizontal Each Quant Tech
  for (row_each, col_each) in index_border_chief:
    fig.add_shape(type="line",
                  x0=-0.5, y0=row_each-0.5, x1=col-0.5, y1=row_each-0.5,
                  line=dict(color="black", width = 1),
                  )
  if show_subline:
    if not display_mean:
      family_counter = list(reversed(family_counter))
    # Horizontal Each Family Language Evaluation
    row_cum = 0
    # for row_each in family_counter:
    for row_each in family_counter * len(base_models):
      row_cum += row_each
      fig.add_shape(type="line",
                    x0=-0.5, y0=row_cum-0.5, x1=col-0.5, y1=row_cum-0.5,
                    line=dict(color="black", width = 1),
                    )

  fig.update_xaxes(automargin=False)
  fig.update_yaxes(automargin=False)
  fig.update_layout(
      # title_text=f"{base_model} {custom_title} {eval_dataset} (Max 1.0, +other language improve): Delta acc with Unquantized of <b>{quantization_technique} {bit_width}-bit</b> model",
      xaxis_title_text='Quantization Technique - Calibration Language',
      yaxis_title_text='Base Model - Eval Language',
      margin=dict(t=70, b=10 if write_pdf else 0, l=120, r=110 if display_colorbar else 0)
  )
  fig.update_layout(coloraxis_colorbar_title_side="right")
  if not display_mean:
    fig.update_xaxes(side="top")

  fig.layout.height = 70 + 35 * row + (10 if write_pdf else 0)
  fig.layout.width = 120 + 89 * col + (110 if display_colorbar else 0)

  # Color bar annotation
  if not display_colorbar:
    fig.update_traces(showscale=False)
  if display_colorbar:
    fig.update_layout(
        # margin=dict(r=150),
        # ^^ making a bit of space for the annotation
        annotations=[
            dict(
                text="Z-score",
                font_size=20,
                font_family='Arial',
                # font_color='red',
                textangle=90,
                showarrow=False,
                # ^^ appearance
                xref="paper",
                yref="paper",
                x=1 + (5 + 10 * col)/(fig.layout.margin.l + fig.layout.width + fig.layout.margin.r),
                y=0.5,
                # ^^ position
            ),
        ]
    )

  # fig.update_traces(showscale=False)
  # fig.update_layout(yaxis_visible=False, yaxis_showticklabels=False)

  fig.show()
  # print(fig)

  if write_pdf:
    print(f"Writing to pdf ...")
    fig.write_image(write_pdf)

#### The data

##### Only the random_seed

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf="ablation_floresplus_3bit_vert.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
0,Qwen/Qwen3-8B,gptq,3-bit,English,40.188512,4.329966,24.304464,160.417519,62.901249
5,Qwen/Qwen3-8B,gptq,3-bit,English,40.949766,4.233635,25.688953,160.373092,64.743586
6,Qwen/Qwen3-8B,gptq,3-bit,English,40.436502,4.268332,24.618706,166.343463,63.556982
2,Qwen/Qwen3-8B,gptq,3-bit,Tamil,42.012760,2.919523,22.620578,144.263196,62.475377
9,Qwen/Qwen3-8B,gptq,3-bit,Tamil,42.729569,2.937405,23.351523,141.058965,65.202294
10,Qwen/Qwen3-8B,gptq,3-bit,Tamil,43.151184,2.916437,23.161336,140.340460,64.788691
1,Qwen/Qwen3-8B,gptq,3-bit,Indonesian,42.385494,3.915570,19.171946,143.357820,62.972576
7,Qwen/Qwen3-8B,gptq,3-bit,Indonesian,43.250695,3.672597,18.982422,133.669715,62.093137
8,Qwen/Qwen3-8B,gptq,3-bit,Indonesian,43.147129,3.778896,19.069444,132.905051,63.068583


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
1,Qwen/Qwen3-8B,tacq,English,3-bit,0.151631,0.039473,0.103480,2.745665,0.322409
4,Qwen/Qwen3-8B,tacq,Tamil,3-bit,0.787157,0.004207,0.143836,1.245594,0.341394
2,Qwen/Qwen3-8B,tacq,Indonesian,3-bit,0.473008,0.012477,0.037528,5.156465,0.841770
3,Qwen/Qwen3-8B,tacq,Swahili,3-bit,0.002507,0.024800,0.039048,0.484618,0.440133
0,Qwen/Qwen3-8B,tacq,Chinese,3-bit,0.376958,0.038915,0.161324,2.276247,0.174826


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
1,Qwen/Qwen3-8B,gptq,English,3-bit,1.607722,0.631171,1.395688,2.210527,1.804371
4,Qwen/Qwen3-8B,gptq,Tamil,3-bit,1.629727,0.466045,1.362567,2.151944,1.807234
2,Qwen/Qwen3-8B,gptq,Indonesian,3-bit,1.632738,0.578527,1.280456,2.135591,1.797347
3,Qwen/Qwen3-8B,gptq,Swahili,3-bit,1.644914,0.571095,1.347168,1.898244,1.813504
0,Qwen/Qwen3-8B,gptq,Chinese,3-bit,1.612060,0.622343,1.395127,2.209036,1.747992


...

## Shape
df text: (5, 16)
df std: (5, 16)
Display df


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Qwen3,46.980806,63.733939,64.155454,62.711432,65.088417,55.974740,64.712779,63.478865,64.819501,68.373666,55.552981,53.220909,54.118815,53.634716,53.881667,48.014369
SW_Qwen3,64.534645,162.378025,141.887540,136.644195,79.112232,161.821596,164.318719,145.458323,137.722535,74.107588,179.623959,118.216036,104.766815,103.911624,65.999048,115.434023
ID_Qwen3,16.699008,24.870707,23.044479,19.074604,22.241715,24.838596,26.285258,23.784272,18.983927,22.583907,27.751146,21.206783,19.529220,17.453221,19.180347,21.117769
TA_Qwen3,2.741605,4.277311,2.924455,3.789021,3.724733,4.191249,4.949053,2.907458,4.009722,3.971942,4.454546,3.795672,2.791451,3.444308,3.318586,3.607402
EN_Qwen3,33.428698,40.524927,42.631171,42.927773,44.148314,40.931761,38.188459,46.080950,41.845462,44.450209,43.978831,34.863418,37.812257,36.722366,37.237706,36.611705


Display df std


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Qwen3,,0.76,1.20,0.44,0.92,0.70,1.57,0.97,0.70,0.92,0.82,0.32,0.34,0.84,0.44,0.17
SW_Qwen3,,2.80,1.71,4.76,1.14,11.74,5.46,7.63,3.36,0.71,8.25,2.75,1.25,5.16,0.48,2.28
ID_Qwen3,,0.59,0.31,0.08,0.42,0.07,0.23,0.53,0.03,0.34,1.07,0.10,0.14,0.04,0.04,0.16
TA_Qwen3,,0.04,0.01,0.10,0.04,0.10,0.09,0.01,0.04,0.02,0.20,0.04,0.00,0.01,0.02,0.04
EN_Qwen3,,0.32,0.47,0.39,1.01,0.25,0.27,1.34,0.68,0.87,1.06,0.15,0.79,0.47,0.00,0.38


Writing to pdf ...


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}, {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}]}},
                            # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf="ablation_floresplus_2bit_vert.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
0,Qwen/Qwen3-8B,gptq,2-bit,English,13062.876244,48256.105485,39854.463930,38231.523295,34286.236528
10,Qwen/Qwen3-8B,gptq,2-bit,English,12451.018302,40756.553633,38410.667460,41726.485947,39319.365601
11,Qwen/Qwen3-8B,gptq,2-bit,English,12969.810741,48809.762083,39445.203775,41236.898022,44868.795841
...,...,...,...,...,...,...,...,...,...
26,CohereLabs/aya-expanse-8b,tacq,2-bit,Swahili,98.850604,10.082932,203.732166,333.068629,407.265983
27,CohereLabs/aya-expanse-8b,tacq,2-bit,Swahili,95.199483,10.107628,198.644097,320.552778,400.672744
9,CohereLabs/aya-expanse-8b,tacq,2-bit,Chinese,87.566569,11.085941,287.207312,2165.595526,136.307643
28,CohereLabs/aya-expanse-8b,tacq,2-bit,Chinese,86.254922,12.776039,322.690826,2640.984867,135.847607


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,tacq,English,2-bit,0.323566,0.665306,3.515737,10.556345,3.317178
9,Qwen/Qwen3-8B,tacq,Tamil,2-bit,0.461728,0.010456,14.028029,144.154089,2.312004
7,Qwen/Qwen3-8B,tacq,Indonesian,2-bit,0.534628,0.818386,0.167729,23.810479,4.750264
8,Qwen/Qwen3-8B,tacq,Swahili,2-bit,0.458314,0.499177,0.602931,1.532377,10.103402
5,Qwen/Qwen3-8B,tacq,Chinese,2-bit,0.907797,1.544423,5.781454,15.010444,0.752618
1,CohereLabs/aya-expanse-8b,tacq,English,2-bit,0.218186,1.433973,4.411970,233.931693,12.841869
4,CohereLabs/aya-expanse-8b,tacq,Tamil,2-bit,1.176844,0.012800,9.131215,52.022620,5.145499
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,2-bit,0.472105,0.360203,2.482059,161.906780,15.790946
3,CohereLabs/aya-expanse-8b,tacq,Swahili,2-bit,1.532074,0.192958,2.532162,8.991836,4.074086
0,CohereLabs/aya-expanse-8b,tacq,Chinese,2-bit,1.514478,0.978149,17.780479,306.709847,0.387395


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,gptq,English,2-bit,4.108156,4.662199,4.593693,4.606363,4.596503
9,Qwen/Qwen3-8B,gptq,Tamil,2-bit,4.642051,0.873260,4.664672,4.670482,4.531575
7,Qwen/Qwen3-8B,gptq,Indonesian,2-bit,4.544935,4.508749,3.226292,4.541529,4.624799
8,Qwen/Qwen3-8B,gptq,Swahili,2-bit,4.643958,4.572244,4.622783,3.166420,4.405375
5,Qwen/Qwen3-8B,gptq,Chinese,2-bit,4.489204,4.365599,4.580982,4.616916,4.249957


...

## Shape
df text: (10, 16)
df std: (10, 16)
Display df


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Aya,112.084228,19168.145370,9278.960985,9950.510277,14767.553330,1029.857317,7.409755e+06,NaN,1.631137e+06,1.004871e+06,274.084188,360.637008,275.617452,292.760948,406.131890,135.838040
SW_Aya,347.245294,22225.967596,21938.558958,14221.136656,1591.607966,20855.464249,9.978332e+03,NaN,5.561333e+03,4.189173e+02,50822.574116,1588.158444,1544.971890,1344.702942,332.042594,2235.354059
ID_Aya,92.363250,6090.955316,6473.671674,545.811702,4321.037729,6165.939366,4.766603e+04,NaN,2.328256e+02,1.044489e+03,484718.872812,306.205275,225.322198,117.137641,202.212119,297.658705
TA_Aya,3.798334,22325.499442,6.436282,4200.197925,1501.481445,4226.914393,4.110211e+05,NaN,4.490107e+05,6.071352e+05,290134.141033,14.739504,4.049754,12.200776,10.231536,12.421246
EN_Aya,53.754913,241.836277,745.679240,522.582423,824.976137,1006.574886,1.366964e+02,NaN,2.688014e+02,3.345557e+02,4622.001841,69.141747,93.205527,80.995250,97.275513,85.909020
ZH_Qwen3,46.980806,39491.465990,34007.536689,42150.094142,25431.655621,17781.024879,6.674189e+05,60166.327417,4.113642e+05,6.278313e+05,360.020385,146.704354,119.462250,122.105214,176.318815,63.072371
SW_Qwen3,64.534645,40398.302421,46825.504463,34796.007236,1466.964319,41391.915611,6.726278e+03,6525.812005,2.713585e+03,5.398065e+02,19233.229068,380.075948,433.095598,258.430765,69.236501,372.747064
ID_Qwen3,16.699008,39236.778388,46203.199536,1683.806307,41954.948059,38104.965297,2.493904e+03,11464.605865,1.232482e+02,7.012918e+02,18727.617100,116.218561,72.677371,21.817674,43.733264,127.139610
TA_Qwen3,2.741605,45940.807067,7.468949,32266.297200,37346.013068,23205.924496,1.476803e+04,4.442923,1.200714e+04,6.139446e+03,25621.911270,15.513642,3.063271,9.956398,9.944748,13.166703
EN_Qwen3,33.428698,12827.901762,43858.254034,35069.944932,44051.236215,30846.394995,1.515746e+02,6890.433754,1.077302e+03,2.003619e+03,5742.270277,43.983318,64.369847,52.554869,65.558487,59.187938


Display df std


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Aya,,1.5e+3,1.3e+3,866.1,1.2e+3,18.95,1.2e+6,,4.2e+5,6.5e+5,13.88,12.84,5.15,15.79,4.07,0.39
SW_Aya,,1.3e+3,2.1e+3,1.2e+3,33.02,2.3e+3,2.1e+3,,1.1e+3,37.52,2.6e+4,233.9,52.02,161.9,8.99,306.7
ID_Aya,,785.6,453.4,36.47,598.0,748.5,1.6e+4,,10.86,407.5,4.7e+4,4.41,9.13,2.48,2.53,17.78
TA_Aya,,6.7e+3,0.24,1.0e+3,576.6,1.1e+3,2.8e+5,,3.2e+5,8.3e+5,2.7e+5,1.43,0.01,0.36,0.19,0.98
EN_Aya,,6.35,60.78,25.46,80.77,75.66,8.65,,22.66,117.2,3.5e+3,0.22,1.18,0.47,1.53,1.51
ZH_Qwen3,,4.3e+3,1.3e+4,1.8e+3,214.6,1.5e+3,2.6e+5,2.3e+4,1.4e+5,5.6e+5,44.89,3.32,2.31,4.75,10.10,0.75
SW_Qwen3,,1.5e+3,41.81,3.2e+3,119.3,6.2e+3,915.6,1.3e+3,1.2e+3,152.9,6.1e+3,10.56,144.2,23.81,1.53,15.01
ID_Qwen3,,607.6,5.9e+3,463.5,2.1e+3,4.6e+3,434.5,2.2e+3,64.11,63.69,5.2e+3,3.52,14.03,0.17,0.60,5.78
TA_Qwen3,,3.7e+3,0.42,3.5e+3,1.5e+3,1.8e+3,9.4e+3,0.10,1.1e+4,4.1e+3,1.6e+4,0.67,0.01,0.82,0.50,1.54
EN_Qwen3,,269.2,3.1e+3,1.5e+3,1.6e+3,492.5,15.20,2.6e+3,570.5,812.8,2.3e+3,0.32,0.46,0.53,0.46,0.91


Writing to pdf ...


##### All data

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [4]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf="ablation_floresplus_4bit_all_vert_final.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
0,Qwen/Qwen3-8B,gptq,4-bit,English,38.633773,2.903144,18.795383,81.519027,54.471988
2,Qwen/Qwen3-8B,gptq,4-bit,Tamil,36.874922,2.789352,18.045387,80.639509,52.047636
1,Qwen/Qwen3-8B,gptq,4-bit,Indonesian,36.307482,2.869462,17.537566,75.503067,52.100443
3,Qwen/Qwen3-8B,gptq,4-bit,Swahili,36.782606,2.862192,18.727565,71.522493,52.984904
4,Qwen/Qwen3-8B,gptq,4-bit,Chinese,36.791005,2.921814,17.943838,78.501095,52.133661
5,CohereLabs/aya-expanse-8b,gptq,4-bit,English,50.815876,3.947913,89.978133,401.770728,113.312037
7,CohereLabs/aya-expanse-8b,gptq,4-bit,Tamil,51.355844,3.796851,90.784244,373.873691,112.280232


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,tacq,English,4-bit,0.0,0.0,0.0,0.0,0.0
9,Qwen/Qwen3-8B,tacq,Tamil,4-bit,0.0,0.0,0.0,0.0,0.0
7,Qwen/Qwen3-8B,tacq,Indonesian,4-bit,0.0,0.0,0.0,0.0,0.0
8,Qwen/Qwen3-8B,tacq,Swahili,4-bit,0.0,0.0,0.0,0.0,0.0
5,Qwen/Qwen3-8B,tacq,Chinese,4-bit,0.0,0.0,0.0,0.0,0.0
1,CohereLabs/aya-expanse-8b,tacq,English,4-bit,0.0,0.0,0.0,0.0,0.0
4,CohereLabs/aya-expanse-8b,tacq,Tamil,4-bit,0.0,0.0,0.0,0.0,0.0
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,4-bit,0.0,0.0,0.0,0.0,0.0
3,CohereLabs/aya-expanse-8b,tacq,Swahili,4-bit,0.0,0.0,0.0,0.0,0.0
0,CohereLabs/aya-expanse-8b,tacq,Chinese,4-bit,0.0,0.0,0.0,0.0,0.0


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,gptq,English,4-bit,1.586967,0.462869,1.274051,1.911259,1.736173
9,Qwen/Qwen3-8B,gptq,Tamil,4-bit,1.566731,0.445503,1.256366,1.906548,1.716401
7,Qwen/Qwen3-8B,gptq,Indonesian,4-bit,1.559996,0.457800,1.243969,1.877965,1.716841
8,Qwen/Qwen3-8B,gptq,Swahili,4-bit,1.565642,0.456699,1.272481,1.854443,1.724152
5,Qwen/Qwen3-8B,gptq,Chinese,4-bit,1.565742,0.465653,1.253915,1.894876,1.717118


...

## Shape
df text: (15, 16)
df std: (15, 16)
Display df


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,40.062125,39.947466,40.115798,40.419192,38.331823,41.523714,NaN,44.224801,46.296442,38.107827,40.119780,39.193255,40.222063,39.940935,37.424322
SW_Llama,30.056285,35.268858,34.343137,33.018261,31.212913,34.342650,36.902012,NaN,37.360703,32.886215,35.678429,34.456392,33.305476,32.882074,30.807823,33.288619
ID_Llama,24.823783,27.111507,27.035368,25.609484,26.443130,27.025912,28.568594,NaN,26.986952,28.200670,27.777874,27.187312,27.168395,25.944766,26.501118,27.072293
TA_Llama,2.311781,2.503244,2.341632,2.454821,2.453562,2.436780,2.695015,NaN,2.735899,2.699435,2.555277,2.503889,2.345541,2.457776,2.460396,2.424910
EN_Llama,45.105214,47.372066,46.727688,46.340421,46.679481,46.697379,43.532462,NaN,44.632579,45.423450,44.928034,45.535295,46.511978,46.849139,46.166003,45.672605
ZH_Aya,112.084228,113.312037,112.280232,113.168497,112.105412,113.046247,124.485867,NaN,122.105721,123.359929,117.615823,120.301677,120.618816,119.716704,121.515856,116.573491
SW_Aya,347.245294,401.770728,373.873691,365.351107,352.421007,381.202734,397.648200,NaN,372.462067,337.157383,412.249550,388.558360,389.291678,381.536862,364.872794,411.201310
ID_Aya,92.363250,89.978133,90.784244,90.535852,89.279740,92.005672,98.354503,NaN,93.870980,97.055222,98.401795,96.781066,97.586160,95.089391,97.101196,96.739712
TA_Aya,3.798334,3.947913,3.796851,3.977342,3.900542,3.929984,4.021575,NaN,3.975950,3.920454,3.954161,3.980362,3.848132,3.942369,3.988709,3.928528
EN_Aya,53.754913,50.815876,51.355844,52.137953,50.556806,52.413353,54.706773,NaN,55.686326,55.946115,55.432520,55.022220,55.689060,55.559975,55.820594,55.483941


Display df std


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,,,,,,,,,,
SW_Llama,,,,,,,,,,,,,,,,
ID_Llama,,,,,,,,,,,,,,,,
TA_Llama,,,,,,,,,,,,,,,,
EN_Llama,,,,,,,,,,,,,,,,
ZH_Aya,,,,,,,,,,,,,,,,
SW_Aya,,,,,,,,,,,,,,,,
ID_Aya,,,,,,,,,,,,,,,,
TA_Aya,,,,,,,,,,,,,,,,
EN_Aya,,,,,,,,,,,,,,,,


Writing to pdf ...


In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf="ablation_floresplus_3bit_all_vert_final.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
0,Qwen/Qwen3-8B,gptq,3-bit,English,40.188512,4.329966,24.304464,160.417519,62.901249
15,Qwen/Qwen3-8B,gptq,3-bit,English,40.949766,4.233635,25.688953,160.373092,64.743586
...,...,...,...,...,...,...,...,...,...
10,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,English,48.270659,4.269030,39.772587,56.283017,57.113641
12,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,Tamil,50.732388,2.430107,35.062178,46.878139,48.481814
11,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,Indonesian,50.051469,3.463090,28.618159,43.463953,52.050964
13,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,Swahili,51.241933,3.554056,34.721147,34.765476,56.756861


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,tacq,English,3-bit,0.151631,0.039473,0.103480,2.745665,0.322409
9,Qwen/Qwen3-8B,tacq,Tamil,3-bit,0.787157,0.004207,0.143836,1.245594,0.341394
7,Qwen/Qwen3-8B,tacq,Indonesian,3-bit,0.473008,0.012477,0.037528,5.156465,0.841770
8,Qwen/Qwen3-8B,tacq,Swahili,3-bit,0.002507,0.024800,0.039048,0.484618,0.440133
5,Qwen/Qwen3-8B,tacq,Chinese,3-bit,0.376958,0.038915,0.161324,2.276247,0.174826
1,CohereLabs/aya-expanse-8b,tacq,English,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
4,CohereLabs/aya-expanse-8b,tacq,Tamil,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
3,CohereLabs/aya-expanse-8b,tacq,Swahili,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
0,CohereLabs/aya-expanse-8b,tacq,Chinese,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,gptq,English,3-bit,1.607722,0.631171,1.395688,2.210527,1.804371
9,Qwen/Qwen3-8B,gptq,Tamil,3-bit,1.629727,0.466045,1.362567,2.151944,1.807234
7,Qwen/Qwen3-8B,gptq,Indonesian,3-bit,1.632738,0.578527,1.280456,2.135591,1.797347
8,Qwen/Qwen3-8B,gptq,Swahili,3-bit,1.644914,0.571095,1.347168,1.898244,1.813504
5,Qwen/Qwen3-8B,gptq,Chinese,3-bit,1.612060,0.622343,1.395127,2.209036,1.747992


...

## Shape
df text: (15, 16)
df std: (15, 16)
Display df


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,65.906926,57.598253,62.209873,62.640825,45.390110,107.699684,896093.535485,68.901162,493.790219,44.496971,57.113641,48.481814,52.050964,56.756861,41.277322
SW_Llama,30.056285,76.299433,63.393429,54.521151,36.941053,70.289028,140.046129,82843.632618,64.677236,127.097711,79.511096,56.283017,46.878139,43.463953,34.765476,50.868524
ID_Llama,24.823783,49.034587,42.820579,32.114254,38.430521,45.473741,80.624982,147212.490658,33.437626,237.449086,52.094113,39.772587,35.062178,28.618159,34.721147,35.462255
TA_Llama,2.311781,4.619366,2.477804,3.813212,3.841755,3.669907,6.090120,349753.537060,4.954544,8.438959,3.885786,4.269030,2.430107,3.463090,3.554056,3.123373
EN_Llama,45.105214,54.002999,56.975352,56.643278,55.005956,56.127699,51.661761,20302.152824,52.608449,82.705680,55.081591,48.270659,50.732388,50.051469,51.241933,50.027819
ZH_Aya,112.084228,170.108220,262.474523,151.644070,153.626228,124.030659,153.807992,NaN,148.348093,152.570887,127.689433,132.582898,135.092829,132.251605,139.732992,119.321911
SW_Aya,347.245294,701.944022,866.680681,601.327532,386.510893,628.567549,694.409390,NaN,581.597703,372.344141,698.011042,488.415269,494.785107,505.833186,351.026762,561.057641
ID_Aya,92.363250,122.900772,170.286285,100.952621,109.504752,108.384593,128.241772,NaN,106.431040,117.153934,123.701714,104.218850,107.600928,96.903556,105.485363,106.001720
TA_Aya,3.798334,5.489648,3.942703,4.941318,4.902821,5.048185,5.190608,NaN,4.846758,4.707457,4.957356,4.598233,3.914037,4.432267,4.394636,4.441016
EN_Aya,53.754913,55.716955,59.996131,58.331153,56.882036,56.490654,70.334848,NaN,70.159288,71.447728,71.276244,56.603639,59.778306,57.988980,59.710328,58.374838


Display df std


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,,,,,,,,,,
SW_Llama,,,,,,,,,,,,,,,,
ID_Llama,,,,,,,,,,,,,,,,
TA_Llama,,,,,,,,,,,,,,,,
EN_Llama,,,,,,,,,,,,,,,,
ZH_Aya,,,,,,,,,,,,,,,,
SW_Aya,,,,,,,,,,,,,,,,
ID_Aya,,,,,,,,,,,,,,,,
TA_Aya,,,,,,,,,,,,,,,,
EN_Aya,,,,,,,,,,,,,,,,


Writing to pdf ...


In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}, {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}]}},
                            # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf="ablation_floresplus_2bit_all_vert_final.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
0,Qwen/Qwen3-8B,gptq,2-bit,English,13062.876244,48256.105485,39854.463930,38231.523295,34286.236528
15,Qwen/Qwen3-8B,gptq,2-bit,English,12451.018302,40756.553633,38410.667460,41726.485947,39319.365601
...,...,...,...,...,...,...,...,...,...
10,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,English,71.305146,98.906951,409.737624,422.779281,698.255733
12,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Tamil,123.015885,2.790848,254.872068,329.592691,276.418416
11,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Indonesian,94.744345,11.290383,48.682029,208.366909,381.739075
13,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Swahili,108.396437,15.895496,135.912888,43.170179,1156.817653


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,tacq,English,2-bit,0.323566,0.665306,3.515737,10.556345,3.317178
9,Qwen/Qwen3-8B,tacq,Tamil,2-bit,0.461728,0.010456,14.028029,144.154089,2.312004
7,Qwen/Qwen3-8B,tacq,Indonesian,2-bit,0.534628,0.818386,0.167729,23.810479,4.750264
8,Qwen/Qwen3-8B,tacq,Swahili,2-bit,0.458314,0.499177,0.602931,1.532377,10.103402
5,Qwen/Qwen3-8B,tacq,Chinese,2-bit,0.907797,1.544423,5.781454,15.010444,0.752618
1,CohereLabs/aya-expanse-8b,tacq,English,2-bit,0.218186,1.433973,4.411970,233.931693,12.841869
4,CohereLabs/aya-expanse-8b,tacq,Tamil,2-bit,1.176844,0.012800,9.131215,52.022620,5.145499
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,2-bit,0.472105,0.360203,2.482059,161.906780,15.790946
3,CohereLabs/aya-expanse-8b,tacq,Swahili,2-bit,1.532074,0.192958,2.532162,8.991836,4.074086
0,CohereLabs/aya-expanse-8b,tacq,Chinese,2-bit,1.514478,0.978149,17.780479,306.709847,0.387395


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
6,Qwen/Qwen3-8B,gptq,English,2-bit,4.108156,4.662199,4.593693,4.606363,4.596503
9,Qwen/Qwen3-8B,gptq,Tamil,2-bit,4.642051,0.873260,4.664672,4.670482,4.531575
7,Qwen/Qwen3-8B,gptq,Indonesian,2-bit,4.544935,4.508749,3.226292,4.541529,4.624799
8,Qwen/Qwen3-8B,gptq,Swahili,2-bit,4.643958,4.572244,4.622783,3.166420,4.405375
5,Qwen/Qwen3-8B,gptq,Chinese,2-bit,4.489204,4.365599,4.580982,4.616916,4.249957


...

## Shape
df text: (15, 16)
df std: (15, 16)
Display df


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,39034.718763,31721.858572,36546.410226,46577.700046,3135.019061,1.182826e+07,NaN,1.889792e+06,3.922542e+06,2407.566671,698.255733,276.418416,381.739075,1156.817653,68.792080
SW_Llama,30.056285,34246.515720,46302.289195,19466.319055,4066.321032,42876.669741,4.843673e+04,NaN,3.387789e+04,6.797139e+04,20333.921389,422.779281,329.592691,208.366909,43.170179,469.148661
ID_Llama,24.823783,31939.373115,38846.104053,2767.984113,27130.168106,35602.009931,9.034067e+04,NaN,8.363805e+04,1.253635e+05,54669.471081,409.737624,254.872068,48.682029,135.912888,481.666138
TA_Llama,2.311781,NaN,9.525656,NaN,NaN,57354.934756,1.325875e+07,NaN,2.981072e+06,5.918960e+05,212825.012183,98.906951,2.790848,11.290383,15.895496,8.777216
EN_Llama,45.105214,1300.394068,8477.161407,6123.151827,10477.869747,7680.062329,1.310330e+03,NaN,2.464687e+04,4.703085e+04,6997.865099,71.305146,123.015885,94.744345,108.396437,110.839197
ZH_Aya,112.084228,19168.145370,9278.960985,9950.510277,14767.553330,1029.857317,7.409755e+06,NaN,1.631137e+06,1.004871e+06,274.084188,360.637008,275.617452,292.760948,406.131890,135.838040
SW_Aya,347.245294,22225.967596,21938.558958,14221.136656,1591.607966,20855.464249,9.978332e+03,NaN,5.561333e+03,4.189173e+02,50822.574116,1588.158444,1544.971890,1344.702942,332.042594,2235.354059
ID_Aya,92.363250,6090.955316,6473.671674,545.811702,4321.037729,6165.939366,4.766603e+04,NaN,2.328256e+02,1.044489e+03,484718.872812,306.205275,225.322198,117.137641,202.212119,297.658705
TA_Aya,3.798334,22325.499442,6.436282,4200.197925,1501.481445,4226.914393,4.110211e+05,NaN,4.490107e+05,6.071352e+05,290134.141033,14.739504,4.049754,12.200776,10.231536,12.421246
EN_Aya,53.754913,241.836277,745.679240,522.582423,824.976137,1006.574886,1.366964e+02,NaN,2.688014e+02,3.345557e+02,4622.001841,69.141747,93.205527,80.995250,97.275513,85.909020


Display df std


,UQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,,,,,,,,,,
SW_Llama,,,,,,,,,,,,,,,,
ID_Llama,,,,,,,,,,,,,,,,
TA_Llama,,,,,,,,,,,,,,,,
EN_Llama,,,,,,,,,,,,,,,,
ZH_Aya,,1.5e+3,1.3e+3,866.1,1.2e+3,18.95,1.2e+6,,4.2e+5,6.5e+5,13.88,12.84,5.15,15.79,4.07,0.39
SW_Aya,,1.3e+3,2.1e+3,1.2e+3,33.02,2.3e+3,2.1e+3,,1.1e+3,37.52,2.6e+4,233.9,52.02,161.9,8.99,306.7
ID_Aya,,785.6,453.4,36.47,598.0,748.5,1.6e+4,,10.86,407.5,4.7e+4,4.41,9.13,2.48,2.53,17.78
TA_Aya,,6.7e+3,0.24,1.0e+3,576.6,1.1e+3,2.8e+5,,3.2e+5,8.3e+5,2.7e+5,1.43,0.01,0.36,0.19,0.98
EN_Aya,,6.35,60.78,25.46,80.77,75.66,8.65,,22.66,117.2,3.5e+3,0.22,1.18,0.47,1.53,1.51


Writing to pdf ...


##### All data - Each Quant

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  for quantization_technique in ["gptq", "slimllm", "tacq"]:
    runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                    filters={"$or": [
                            { "$and": [
                              {"config.quantization_technique": quantization_technique.lower()},
                              {"config.bit_width": f"{bit_width}-bit"},
                              {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                            ]
                            },
                            { "$and" : [
                                {"config.quantization_technique": "Unquantized"},
                                # {"config.evaluation_dataset": evaluation_dataset},
                                {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                                # {"config.base_model": base_model},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                                {"config.enable_thinking": False},
                                {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                            ]}
                        ]}
                    )

    eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
    # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
    # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
    # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
    # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
    # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

    # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
    perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf=f"ablation_floresplus_{bit_width}bit_{quantization_technique}_all_vert.pdf")

/tmp/ipykernel_9034/1384151445.py:165: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
0,Qwen/Qwen3-8B,gptq,3-bit,English,40.188512,4.329966,24.304464,160.417519,62.901249
15,Qwen/Qwen3-8B,gptq,3-bit,English,40.949766,4.233635,25.688953,160.373092,64.743586
16,Qwen/Qwen3-8B,gptq,3-bit,English,40.436502,4.268332,24.618706,166.343463,63.556982
2,Qwen/Qwen3-8B,gptq,3-bit,Tamil,42.012760,2.919523,22.620578,144.263196,62.475377
19,Qwen/Qwen3-8B,gptq,3-bit,Tamil,42.729569,2.937405,23.351523,141.058965,65.202294
20,Qwen/Qwen3-8B,gptq,3-bit,Tamil,43.151184,2.916437,23.161336,140.340460,64.788691
1,Qwen/Qwen3-8B,gptq,3-bit,Indonesian,42.385494,3.915570,19.171946,143.357820,62.972576


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


...

## Shape
df text: (15, 6)
df std: (15, 6)
Display df


,UQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,57.113641,48.481814,52.050964,56.756861,41.277322
SW_Llama,30.056285,56.283017,46.878139,43.463953,34.765476,50.868524
ID_Llama,24.823783,39.772587,35.062178,28.618159,34.721147,35.462255
TA_Llama,2.311781,4.269030,2.430107,3.463090,3.554056,3.123373
EN_Llama,45.105214,48.270659,50.732388,50.051469,51.241933,50.027819
ZH_Aya,112.084228,132.582898,135.092829,132.251605,139.732992,119.321911
SW_Aya,347.245294,488.415269,494.785107,505.833186,351.026762,561.057641
ID_Aya,92.363250,104.218850,107.600928,96.903556,105.485363,106.001720
TA_Aya,3.798334,4.598233,3.914037,4.432267,4.394636,4.441016
EN_Aya,53.754913,56.603639,59.778306,57.988980,59.710328,58.374838


Display df std


,UQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,
SW_Llama,,,,,,
ID_Llama,,,,,,
TA_Llama,,,,,,
EN_Llama,,,,,,
ZH_Aya,,,,,,
SW_Aya,,,,,,
ID_Aya,,,,,,
TA_Aya,,,,,,
EN_Aya,,,,,,


Writing to pdf ...


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  for quantization_technique in ["gptq", "slimllm", "tacq"]:
    runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                    filters={"$or": [
                            { "$and": [
                              {"config.quantization_technique": quantization_technique.lower()},
                              {"config.bit_width": f"{bit_width}-bit"},
                              {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"$not": {"$and": [{"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}, {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}]}},
                              # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                            ]
                            },
                            { "$and" : [
                                {"config.quantization_technique": "Unquantized"},
                                # {"config.evaluation_dataset": evaluation_dataset},
                                {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                                # {"config.base_model": base_model},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                                {"config.enable_thinking": False},
                                {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                            ]}
                        ]}
                    )

    eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
    # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
    # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
    # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
    # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
    # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

    # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
    perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, write_pdf=f"ablation_floresplus_{bit_width}bit_{quantization_technique}_all_vert.pdf")

/tmp/ipykernel_9034/1384151445.py:165: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
0,Qwen/Qwen3-8B,gptq,2-bit,English,13062.876244,48256.105485,39854.463930,38231.523295,34286.236528
15,Qwen/Qwen3-8B,gptq,2-bit,English,12451.018302,40756.553633,38410.667460,41726.485947,39319.365601
16,Qwen/Qwen3-8B,gptq,2-bit,English,12969.810741,48809.762083,39445.203775,41236.898022,44868.795841
2,Qwen/Qwen3-8B,gptq,2-bit,Tamil,45011.900151,8.059500,48568.139986,NaN,51742.189476
19,Qwen/Qwen3-8B,gptq,2-bit,Tamil,46991.916079,7.181371,51941.771740,46783.693412,23084.039656
20,Qwen/Qwen3-8B,gptq,2-bit,Tamil,39570.945873,7.165976,38099.686881,46867.315514,27196.380936
1,Qwen/Qwen3-8B,gptq,2-bit,Indonesian,33757.634711,27336.007204,1292.570740,31265.346026,44669.351995


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


...

## Shape
df text: (15, 6)
df std: (15, 6)
Display df


,UQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,698.255733,276.418416,381.739075,1156.817653,68.792080
SW_Llama,30.056285,422.779281,329.592691,208.366909,43.170179,469.148661
ID_Llama,24.823783,409.737624,254.872068,48.682029,135.912888,481.666138
TA_Llama,2.311781,98.906951,2.790848,11.290383,15.895496,8.777216
EN_Llama,45.105214,71.305146,123.015885,94.744345,108.396437,110.839197
ZH_Aya,112.084228,360.637008,275.617452,292.760948,406.131890,135.838040
SW_Aya,347.245294,1588.158444,1544.971890,1344.702942,332.042594,2235.354059
ID_Aya,92.363250,306.205275,225.322198,117.137641,202.212119,297.658705
TA_Aya,3.798334,14.739504,4.049754,12.200776,10.231536,12.421246
EN_Aya,53.754913,69.141747,93.205527,80.995250,97.275513,85.909020


Display df std


,UQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,
SW_Llama,,,,,,
ID_Llama,,,,,,
TA_Llama,,,,,,
EN_Llama,,,,,,
ZH_Aya,,12.84,5.15,15.79,4.07,0.39
SW_Aya,,233.9,52.02,161.9,8.99,306.7
ID_Aya,,4.41,9.13,2.48,2.53,17.78
TA_Aya,,1.43,0.01,0.36,0.19,0.98
EN_Aya,,0.22,1.18,0.47,1.53,1.51


Writing to pdf ...


#### 3x Run + SINQ

In [8]:
import numpy as np
import re
def perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, base_models, quantization_technique, bit_width, eval_dataset, custom_title="", display_mean=True, compile=False, show_subline=True, hf_evaluate=False, zscore_threshold=1, gmean_aggregate=False, two_color=True, display_colorbar=True, embed_sinq=True, write_pdf=""):
  language_sort = {
      # # Flores Plus
      # "eng_Latn": 0, "fra_Latn": 1, "hin_Deva": 2, "npi_Deva": 3, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "cmn_Hans": 12, "jpn_Jpan": 13, "kor_Hang": 14, "arb_Arab" : 15,\
      # Wikipedia
      "en": 0, "fr": 1, "hi": 2, "ne": 3, "ta": 6, "te": 7, "id": 8, "ms": 9, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  } # Add floresplus evals, and count their respective language's family
  family_counter = []
  if eval_dataset == 'floresplus':
    # flores_plus_add = ["afr_Latn", "dan_Latn", "deu_Latn", "eng_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"]
    flores_plus_add = ["eng_Latn", "tpi_Latn", "afr_Latn", "deu_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "ydd_Hebr", "dan_Latn", "fao_Latn", "isl_Latn", "nno_Latn", "nob_Latn", "swe_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 0 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["kan_Knda", "mal_Mlym", "tam_Taml", "tel_Telu"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 6 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    # flores_plus_add = ["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "ind_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"]
    flores_plus_add = ["ind_Latn", "zsm_Latn", "bjn_Latn", "min_Latn", "ace_Latn", "ban_Latn", "bug_Latn", "jav_Latn", "sun_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "fil_Latn", "war_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 8 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "swh_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 10 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant", "cmn_Hans"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 12 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
  elif eval_dataset == 'wikipedia':
    wikipedia_add = ["en", "de", "nl"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 0 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["ml", "ta", "te"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 6 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["id", "tl", "ms"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 8 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["sw", "lg", "yo"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 10 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
    wikipedia_add = ["bo", "my", "zh"]
    for index, lang in enumerate(wikipedia_add):
      language_sort[lang] = 12 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in wikipedia_add]))
  eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))
  # print(eval_languages)

  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language"]
  columns.extend(eval_languages)
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5, 'Sinq': 0.5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3, 'sinq': 0.5,\
                 "Qwen/Qwen3-8B": 0, "CohereLabs/aya-expanse-8b": 1, "meta-llama/Llama-3.1-8B-Instruct": 2, "Qwen/Qwen3-14B": 3}

  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH", "Sinq": "SINQ"}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "CohereLabs/aya-expanse-8b": "Aya", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "Qwen/Qwen3-14B": "Qwen3 14B"}
  quantization_technique_label = {"UQ": "Unquantized", "gptq": "GPTQ", "slimllm": "SliM-LLM", "tacq": "TaCQ", "sinq": "SINQ"}
  eval_label = {
      # # Flores Plus
      "eng_Latn": "EN", "fra_Latn": "FR", "hin_Deva": "HI", "npi_Deva": "NE", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "zsm_Latn": "ZSM", "swh_Latn": "SW", "yor_Latn": "YO", "cmn_Hans": "ZH", "jpn_Jpan": "JA", "kor_Hang": "KO", "arb_Arab" : "AR",\
      "ace_Latn": "ACE", "ban_Latn": "BAN", "bjn_Latn": "BJN", "bug_Latn": "BUG", "ceb_Latn": "CEB", "fij_Latn": "FIJ", "ilo_Latn": "ILO", "jav_Latn": "JAV", "min_Latn": "MIN", "plt_Latn": "PLT", "mri_Latn": "MRI", "pag_Latn": "PAG", "smo_Latn": "SMO", "sun_Latn": "SUN", "fil_Latn": "FIL", "war_Latn": "WAR",\
      # Wikipedia
      "en": "EN", "fr": "FR", "hi": "HI", "ne": "NE", "ta": "TA", "te": "TE", "id": "ID", "ms": "MS", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # # Belebele
      # "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TE", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
  }

  data_acc = []
  data_nancount = []

  data_gptq = []
  data_slimllm = []
  data_tacq = []

  # import numpy as np
  # data_slimllm.append(["-", "CohereLabs/aya-expanse-8b", "slimllm", "4-bit", "Tamil"] + [np.nan for _ in eval_languages])

  unquantized_data_acc = []

  # Iterate over the runs and print details
  for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue
      # Special case for SINQ, treat its calib lang as sinq
      if config["quantization_technique"] == "sinq":
        config["calibration_language"] = "Sinq"

      run_data_acc = []
      run_data_nancount = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      run_data_nancount.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      if not hf_evaluate:
        for lang in eval_languages:
          if lang in ["zh", "Chinese", "cmn_Hans", "ja", "Japanese", "jpn_Jpan", "bo", "Tibetan", "bod_Tibt", "dz", "Dzongkha", "dzo_Tibt", "Yue Chinese", "yue_Hant"]:
            acc = run.summary.get(f"{eval_dataset.lower()}_byte_acc_{lang}", None)
            nancount = run.summary.get(f"{eval_dataset.lower()}_nan_count_{lang}", 0)
            if acc:
              # acc = math.pow(acc, 1/3)
              pass
            run_data_acc.append(acc)
            run_data_nancount.append(nancount)
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_token_perplexity_{lang}",
                                                run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                                ))
            run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nan_count_{lang}", 0))
      elif hf_evaluate:
        for lang in eval_languages:
          if gmean_aggregate:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_gmean_perplexity_{lang}",
                                                run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                                ))
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_token_perplexity_{lang}",
                                                run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None)
                                                ))
          run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nancount_{lang}", 0))
      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "gptq":
        data_gptq.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "slimllm":
        data_slimllm.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "tacq":
        data_tacq.append(run_data_acc)
        # data_nancount.append(run_data_nancount)
      elif config["quantization_technique"] == "sinq":
        run_data_acc[1] = "gptq"
        data_gptq.append(run_data_acc.copy())
        run_data_acc[1] = "slimllm"
        data_slimllm.append(run_data_acc.copy())
        run_data_acc[1] = "tacq"
        data_tacq.append(run_data_acc.copy())

  # make sure sinq is not the only datum in each quant data, check by the calib langs
  if embed_sinq:
    calib_langs = set([datum[3] for datum in data_gptq])
    if len(calib_langs) == 1 and "Sinq" in calib_langs:
      data_gptq = []
    calib_langs = set([datum[3] for datum in data_slimllm])
    if len(calib_langs) == 1 and "Sinq" in calib_langs:
      data_slimllm = []
    calib_langs = set([datum[3] for datum in data_tacq])
    if len(calib_langs) == 1 and "Sinq" in calib_langs:
      data_tacq = []

  df_unquantized_acc_chief = pd.DataFrame(unquantized_data_acc, columns=columns)
  df_gptq_chief = pd.DataFrame(data_gptq, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_slimllm_chief = pd.DataFrame(data_slimllm, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_tacq_chief = pd.DataFrame(data_tacq, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  display(pd.concat([df_unquantized_acc_chief, df_gptq_chief, df_slimllm_chief, df_tacq_chief], axis=0))

  # with pd.option_context('display.max_rows', None):

  # Unquantized
  df_unquantized_acc_chief_std = df_unquantized_acc_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std")

  # GPTQ agg
  df_gptq_chief_std = df_gptq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_gptq_chief = df_gptq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  # Slim agg
  df_slimllm_chief_std = df_slimllm_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_slimllm_chief = df_slimllm_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  # Tacq agg
  df_tacq_chief_std = df_tacq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_tacq_chief = df_tacq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  with pd.option_context('display.max_rows', None):
    display(df_tacq_chief_std)
  display(df_gptq_chief.columns)

  # display(df_gptq_chief_acc)
  # display(df_gptq_chief_std)

  df_gptq_chief[df_gptq_chief.columns[4:]] = df_gptq_chief[df_gptq_chief.columns[4:]].astype(float)
  df_slimllm_chief[df_slimllm_chief.columns[4:]] = df_slimllm_chief[df_slimllm_chief.columns[4:]].astype(float)
  df_tacq_chief[df_tacq_chief.columns[4:]] = df_tacq_chief[df_tacq_chief.columns[4:]].astype(float)
  df_unquantized_acc_chief[df_unquantized_acc_chief.columns[4:]] = df_unquantized_acc_chief[df_unquantized_acc_chief.columns[4:]].astype(float)

  # Easying aggregate calculation
  # Perplexity related, change order of PPL
  df_gptq_chief.iloc[:, 4:] = df_gptq_chief.iloc[:, 4:].apply(np.log10)
  df_slimllm_chief.iloc[:, 4:] = df_slimllm_chief.iloc[:, 4:].apply(np.log10)
  df_tacq_chief.iloc[:, 4:] = df_tacq_chief.iloc[:, 4:].apply(np.log10)
  df_unquantized_acc_chief.iloc[:, 4:] = df_unquantized_acc_chief.iloc[:, 4:].apply(np.log10)


  fig = go.Figure()

  df_chief = pd.DataFrame([])
  df_chief_std = pd.DataFrame([])
  df_zscore_chief = pd.DataFrame([])
  index_border_chief = [(0, 0)]
  # for base_model in ["CohereLabs/aya-expanse-8b"]:
  for base_model in base_models:
    print(f"start model: {base_model}")
    df_gptq = df_gptq_chief.loc[df_gptq_chief["Base Model"] == base_model]
    df_slimllm = df_slimllm_chief.loc[df_slimllm_chief["Base Model"] == base_model]
    df_tacq = df_tacq_chief.loc[df_tacq_chief["Base Model"] == base_model]
    df_unquantized_acc = df_unquantized_acc_chief.loc[df_unquantized_acc_chief["Base Model"] == base_model]
    df_gptq_std = df_gptq_chief_std.loc[df_gptq_chief_std["Base Model"] == base_model]
    df_slimllm_std = df_slimllm_chief_std.loc[df_slimllm_chief_std["Base Model"] == base_model]
    df_tacq_std = df_tacq_chief_std.loc[df_tacq_chief_std["Base Model"] == base_model]
    df_unquantized_acc_std = df_unquantized_acc_chief_std.loc[df_unquantized_acc_chief_std["Base Model"] == base_model]
    display(df_gptq)

    # display(df_unquantized_acc)
    base_model = base_model_label[base_model]

    df_gptq_std = df_gptq_std.iloc[:, 4:].T.rename(df_gptq_std['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_gptq_std['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_slimllm_std = df_slimllm_std.iloc[:, 4:].T.rename(df_slimllm_std['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_slimllm_std['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_tacq_std = df_tacq_std.iloc[:, 4:].T.rename(df_tacq_std['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_tacq_std['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_unquantized_acc_std = df_unquantized_acc_std.iloc[:, 4:].T.rename(lambda x: "UQ", axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')

    # Take only the result and reformat the indices
    # df_gptq = df_gptq.iloc[:, 5:].sub(df_unquantized_acc.iloc[:, 5:].values).T.rename(df_gptq['Quantization Technique'].apply(lambda x: x[0].upper()) + '-' + df_gptq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_gptq = df_gptq.iloc[:, 4:].T.rename(df_gptq['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_gptq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_gptq = pd.concat([df_gptq.T, df_gptq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_gptq = df_gptq.astype(float)
    display(df_gptq)

    # df_slimllm = df_slimllm.iloc[:, 5:].sub(df_unquantized_acc.iloc[:, 5:].values).T.rename(df_slimllm['Quantization Technique'].apply(lambda x: x[0].upper()) + '-' + df_slimllm['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_slimllm = df_slimllm.iloc[:, 4:].T.rename(df_slimllm['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_slimllm['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_slimllm = pd.concat([df_slimllm.T, df_slimllm.mean(axis=0).to_frame('Mean')], axis=1).T
    df_slimllm = df_slimllm.astype(float)

    # df_tacq = df_tacq.iloc[:, 5:].sub(df_unquantized_acc.iloc[:, 5:].values).T.rename(df_tacq['Quantization Technique'].apply(lambda x: x[0].upper()) + '-' + df_tacq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    df_tacq = df_tacq.iloc[:, 4:].T.rename(df_tacq['Quantization Technique'].apply(lambda x: quantization_technique_label.get(x, x)) + '-' + df_tacq['Calibration Language'].apply(lambda x: calib_label[x]), axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_tacq = pd.concat([df_tacq.T, df_tacq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_tacq = df_tacq.astype(float)

    df_unquantized_acc = df_unquantized_acc.iloc[:, 4:].T.rename(lambda x: "UQ", axis='columns').rename(lambda x: eval_label.get(x, x) + "_" + base_model, axis='index')
    if display_mean:
      df_unquantized_acc = pd.concat([df_unquantized_acc, df_unquantized_acc.mean(axis=0).to_frame(f'Mean_{base_model}').T], axis=0)
    df = pd.concat([df_gptq, df_slimllm, df_tacq], axis=1)
    df_std = pd.concat([df_gptq_std, df_slimllm_std, df_tacq_std], axis=1)
    df_mean_quantized = df.mean(axis=1).to_frame('Mean of Quantized')
    # display(df_unquantized_acc)

    for iter, (index_row, index_col) in enumerate([df_unquantized_acc.shape, df_gptq.shape, df_slimllm.shape, df_tacq.shape]):
      if iter != 0:
        index_row = 0
      index_border_chief.append((index_border_chief[-1][0] + index_row, index_border_chief[-1][1] + index_col))

    from scipy.stats import zscore
    df_zscore_gptq = pd.DataFrame(zscore(df_gptq, axis=1, nan_policy='omit'), columns=df_gptq.columns, index=df_gptq.index)
    df_zscore_slimllm = pd.DataFrame(zscore(df_slimllm, axis=1, nan_policy='omit'), columns=df_slimllm.columns, index=df_slimllm.index)
    df_zscore_tacq = pd.DataFrame(zscore(df_tacq, axis=1, nan_policy='omit'), columns=df_tacq.columns, index=df_tacq.index)

    # Skip mean of quantized
    df_zscore = pd.concat([pd.Series(np.zeros_like(df_zscore_tacq.index), index=df_zscore_tacq.index, name="UQ"), df_zscore_gptq, df_zscore_slimllm, df_zscore_tacq], axis=1)

    df = pd.concat([df_unquantized_acc, df], axis=1)
    df_chief = pd.concat([df_chief, df], axis=0)
    df_zscore_chief = pd.concat([df_zscore_chief, df_zscore], axis=0)

    df_std = pd.concat([df_unquantized_acc_std, df_std], axis=1)
    df_chief_std = pd.concat([df_chief_std, df_std], axis=0)

  # Revariable for onness usage of df, we was using chief for the sake of differentiating between chief and iter df
  df = df_chief
  df_std = df_chief_std
  df_zscore = df_zscore_chief
  df = df.rename(lambda x: eval_label.get(x, x), axis=0)
  df_std = df_std.rename(lambda x: eval_label.get(x, x), axis=0)
  df_zscore = df_zscore.rename(lambda x: eval_label.get(x, x), axis=0)
  # display(df)
  # display(df_zscore)

  # Rebase the number
  df = df.apply(lambda x: 10 ** x)

  # display(df)
  # display(df_zscore)
  if not display_mean:
    df = df.reindex(index=df.index[::-1])
    df_std = df_std.reindex(index=df_std.index[::-1])
    df_zscore = df_zscore.reindex(index=df_zscore.index[::-1])
  display(df)
  display(df_std)

  # df_text = df.copy().map(lambda x: f"x10^{x:.1f}" if not np.isnan(x) else "-")
  # df_text = df.copy().map(lambda x: f"{(10**x):.2f}" if x < 3 else f"{(10**x):.1e}" if not np.isnan(x) else "-")
  df_std = df_std.map(lambda x: f"{x:.2f}" if 0 < x < 100 else f"{x:.1f}" if 0 < x < 1000 else re.sub(r'([eE][+-])0+(?=\d)', r'\1', f"{x:.1e}") if (not np.isnan(x) and not x == 0) else "")
  df_text = df.copy().map(lambda x: f"{x:.2f}" if x < 100 else f"{x:.1f}" if x < 1000 else re.sub(r'([eE][+-])0+(?=\d)', r'\1', f"{x:.1e}") if not np.isnan(x) else "-")
  # df_text = list(zip(df_text.values, df_std.values))
  print("## Shape")
  print(f"df text: {df_text.shape}")
  print(f"df std: {df_std.shape}")
  df_text = [list(zip(x, y)) for x, y in list(zip(df_text.values.tolist(), df_std.values.tolist()))]
  # df_text.iloc[:, 0] = df_text.iloc[:, 0].apply(lambda x: x[1:])

  # Color palette use this: Green: https://colordesigner.io/gradient-generator from 255, 255, 255 to 6, 54, 21, Red: https://colordesigner.io/gradient-generator from 255, 255, 255 to 214, 39, 40
  def exact_zscore_scale(z, df_zscore):
    return (z - df_zscore.min(axis=None)) / (df_zscore.max(axis=None) - df_zscore.min(axis=None))
  # Add 0, then iteratively add green and red until it touch min 0 or max 1
  colorscale = []
  # If light = good, dark = bad
  if not two_color:
    colorscale = [[0, 'rgba(255, 255, 255, 0.85)'], [1, 'rgba(214, 39, 40, 0.85)']]
  # For two color signaling, e.g. red = bad, green = good
  else:
    colorscale = [[exact_zscore_scale(0, df_zscore), 'rgba(255, 255, 255, 0.85)']]
    # Green for negativee z   1. Three scale different color, 2. Normal, 3. Thresholded z-score
    if zscore_threshold >= 1:
      color_zip = zip([-1+10**-5, -1, -2, -3], ['rgba(255, 255, 255, 0.85)', 'rgba(144, 166, 151, 0.85)', 'rgba(61, 99, 73, 0.85)', 'rgba(6,54,21, 0.85)'])
    elif zscore_threshold >= 1/2:
      color_zip = zip([-1/2+10**-5, -1/2, -2, -3], ['rgba(255, 255, 255, 0.85)', 'rgba(144, 166, 151, 0.85)', 'rgba(61, 99, 73, 0.85)', 'rgba(6,54,21, 0.85)'])
    else:
      # color_zip = zip([-1, -2, -3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
      color_zip = zip([-3], ['rgba(6,54,21, 0.85)'])
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    for z, color in color_zip:
    # for z, color in zip([-10**-5, -1, -1-10**-5, -2, -3], ['rgba(0,255,255, 0.85)', 'rgba(0,255,255, 0.85)', 'rgba(6,54,21, 0.85)', 'rgba(6,54,21, 0.85)', 'rgba(191,139,255, 0.85)']):
    # for z, color in zip([-1, -1-10**(-5), -2], ['rgba(144, 166, 151, 0.85)', 'rgba(89, 121, 99, 0.85)', 'rgba(6,54,21, 0.85)']):
    # for z, color in zip([-1+10**-5, -1, -2, -3], ['rgba(255, 255, 255, 0.85)', 'rgba(144, 166, 151, 0.85)', 'rgba(61, 99, 73, 0.85)', 'rgba(6,54,21, 0.85)']):
      z_scale = exact_zscore_scale(z, df_zscore)
      colorscale.insert(0, [max(0, z_scale), color])
      if z_scale < 0:
        break
    # Red for positive z
    if zscore_threshold >= 1:
      color_zip = zip([1-10**-5, 1, 2, 3], ['rgba(255, 255, 255, 0.85)', 'rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
    elif zscore_threshold >= 1/2:
      color_zip = zip([1/2-10**-5, 1/2, 2, 3], ['rgba(255, 255, 255, 0.85)', 'rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
    else:
      # color_zip = zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)'])
      color_zip = zip([3], ['rgba(214, 39, 40, 0.85)'])
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    # for z, color in zip([1, 2, 3], ['rgba(228, 111, 112, 0.85)', 'rgba(223, 87, 88, 0.85)', 'rgba(214, 39, 40, 0.85)']):
    for z, color in color_zip:
      z_scale = exact_zscore_scale(z, df_zscore)
      colorscale.append([min(1, z_scale), color])
      if z_scale > 1:
        break
  print("Display df")
  display(df)
  print("Display df std")
  display(df_std)
  # Helper how much space to differentiate each SINQ
  convert_to_space = {
      "UQ": '',
      "GPTQ": '',
      "SliM": ' ',
      "TaCQ": '  ',
  }
  fig = go.Figure(data=go.Heatmap(
                      z=df_zscore,
                      x=[[f"{convert_to_space.get(col_name.split('-')[0])}SINQ{convert_to_space.get(col_name.split('-')[0])}" if col_name.split("-")[-1] == "SINQ" else col_name.split("-")[0] if col_name != "UQ" else "16-bit" for col_name in df_zscore.columns],
                       ["-" if col_name.split("-")[-1] == "SINQ" else col_name.split("-")[-1] if col_name != "UQ" else "-" for col_name in df_zscore.columns]],
                      # Trying to rotate -90 for y axis, but alas
                      # y=[[f"""<span style='rotate: 0.5turn; display: inline-block;'> {idx_name.split("_")[-1]} </span>""" for idx_name in df_zscore.index],
                      #  [f"""<span style='rotate: -90deg;'> {idx_name.split("_")[-0]} </span>""" for idx_name in df_zscore.index]],
                      y=[[f"""{idx_name.split("_")[-1]}""" for idx_name in df_zscore.index],
                       [f"""{idx_name.split("_")[-0]}""" for idx_name in df_zscore.index]],
                      # text=[list(zip(x, y)) for x, y in list(zip(pd.concat([df_unquantized_acc, df, df_mean_quantized], axis=0).T.values.tolist(), df_nancount_all.T.values.tolist()))],
                      text=df_text,
                      # texttemplate="×10^%{text:.1f}",
                      texttemplate="%{text[0]}<sub>%{text[1]}</sub>",
                      # colorscale='Viridis',
                      # colorscale= [[0, 'rgba(6,54,21, 0.85)'],
                      #               [(0 - df_zscore.min(axis=None)) / (df_zscore.max(axis=None) - df_zscore.min(axis=None)), 'rgba(255, 255, 255, 0.85)'],
                      #               [1, 'rgba(214, 39, 40, 0.85)']],
                      colorscale=colorscale,
                      # colorbar=dict(title='Z-score'),
                      textfont={"size":28 if bit_width == 4 else 22 if bit_width == 3 else 17, "family": "Arial Black"},
                  ))
  # if bit_width == 4:
  fig.update_layout(
      xaxis=dict(title_font=dict(size=18), tickfont=dict(size=18)),
      yaxis=dict(title_font=dict(size=18), tickfont=dict(size=18))
  )
  # print(colorscale)

  # Horizontal each Eval Languages
  row_gptq, col_gptq = df_gptq.shape
  row_slimllm, col_slimllm = df_slimllm.shape
  row_tacq, col_tacq = df_tacq.shape
  row, col = df_zscore.shape
  offset_mean = 0 if display_mean else 1
  # Horizontal Line
  fig.add_shape(type="line",
                x0=-0.5, y0=row+offset_mean-1.5, x1=col-0.5, y1=row+offset_mean-1.5,
                line=dict(color="black", width = 1),
                )
  # # Vertical Line
  # fig.add_shape(type="line",
  #               x0=col-1.5, y0=-0.5, x1=col-1.5, y1=row-1.5,
  #               line=dict(color="black", width = 5),
  #               )
  # Vertical Each Quant Tech
  # print(index_border_chief)
  for (row_each, col_each) in index_border_chief[:5]:
    fig.add_shape(type="line",
                  x0=col_each-0.5, y0=-0.5, x1=col_each-0.5, y1=row+offset_mean-1.5,
                  line=dict(color="black", width = 5),
                  )
  # Horizontal Each Quant Tech
  for (row_each, col_each) in index_border_chief:
    fig.add_shape(type="line",
                  x0=-0.5, y0=row_each-0.5, x1=col-0.5, y1=row_each-0.5,
                  line=dict(color="black", width = 1),
                  )
  if show_subline:
    if not display_mean:
      family_counter = list(reversed(family_counter))
    # Horizontal Each Family Language Evaluation
    row_cum = 0
    # for row_each in family_counter:
    for row_each in family_counter * len(base_models):
      row_cum += row_each
      fig.add_shape(type="line",
                    x0=-0.5, y0=row_cum-0.5, x1=col-0.5, y1=row_cum-0.5,
                    line=dict(color="black", width = 1),
                    )

  fig.update_xaxes(automargin=False)
  fig.update_yaxes(automargin=False)
  fig.update_layout(
      # title_text=f"{base_model} {custom_title} {eval_dataset} (Max 1.0, +other language improve): Delta acc with Unquantized of <b>{quantization_technique} {bit_width}-bit</b> model",
      xaxis_title_text='Quantization Technique - Calibration Language',
      yaxis_title_text='Base Model - Eval Language',
      margin=dict(t=85, b=10 if write_pdf else 0, l=160, r=110 if display_colorbar else 0)
  ) # top 70 != 4bit else 90, left 120 != 4bit else 165
  fig.update_layout(coloraxis_colorbar_title_side="right")
  if not display_mean:
    fig.update_xaxes(side="top")

  fig.layout.height = 85 + 35 * row + (10 if write_pdf else 0)
  fig.layout.width = 160 + 89 * col + (110 if display_colorbar else 0)

  # Color bar annotation
  if not display_colorbar:
    fig.update_traces(showscale=False)
  if display_colorbar:
    fig.update_layout(
        # margin=dict(r=150),
        # ^^ making a bit of space for the annotation
        annotations=[
            dict(
                text="Z-score",
                font_size=20,
                font_family='Arial',
                # font_color='red',
                textangle=90,
                showarrow=False,
                # ^^ appearance
                xref="paper",
                yref="paper",
                x=1 + (5 + 10 * col)/(fig.layout.margin.l + fig.layout.width + fig.layout.margin.r),
                y=0.5,
                # ^^ position
            ),
        ]
    )

  # fig.update_traces(showscale=False)
  # fig.update_layout(yaxis_visible=False, yaxis_showticklabels=False)

  fig.show()
  # print(fig)

  if write_pdf:
    print(f"Writing to pdf ...")
    fig.write_image(write_pdf)

##### All data

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [4]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            # {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"config.bit_width": f"{bit_width}-bit"},
                            # {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            # {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_{bit_width}bit_sinq_all_vert_final.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
15,Qwen/Qwen3-8B,gptq,4-bit,Sinq,35.693894,2.847228,17.717172,73.394102,52.279770
0,Qwen/Qwen3-8B,gptq,4-bit,English,38.633773,2.903144,18.795383,81.519027,54.471988
2,Qwen/Qwen3-8B,gptq,4-bit,Tamil,36.874922,2.789352,18.045387,80.639509,52.047636
1,Qwen/Qwen3-8B,gptq,4-bit,Indonesian,36.307482,2.869462,17.537566,75.503067,52.100443
3,Qwen/Qwen3-8B,gptq,4-bit,Swahili,36.782606,2.862192,18.727565,71.522493,52.984904
4,Qwen/Qwen3-8B,gptq,4-bit,Chinese,36.791005,2.921814,17.943838,78.501095,52.133661
16,CohereLabs/aya-expanse-8b,gptq,4-bit,Sinq,55.876352,4.002676,98.200647,394.071986,122.265194


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
9,Qwen/Qwen3-8B,tacq,Sinq,4-bit,0.0,0.0,0.0,0.0,0.0
7,Qwen/Qwen3-8B,tacq,English,4-bit,0.0,0.0,0.0,0.0,0.0
11,Qwen/Qwen3-8B,tacq,Tamil,4-bit,0.0,0.0,0.0,0.0,0.0
8,Qwen/Qwen3-8B,tacq,Indonesian,4-bit,0.0,0.0,0.0,0.0,0.0
10,Qwen/Qwen3-8B,tacq,Swahili,4-bit,0.0,0.0,0.0,0.0,0.0
6,Qwen/Qwen3-8B,tacq,Chinese,4-bit,0.0,0.0,0.0,0.0,0.0
3,CohereLabs/aya-expanse-8b,tacq,Sinq,4-bit,0.0,0.0,0.0,0.0,0.0
1,CohereLabs/aya-expanse-8b,tacq,English,4-bit,0.0,0.0,0.0,0.0,0.0
5,CohereLabs/aya-expanse-8b,tacq,Tamil,4-bit,0.0,0.0,0.0,0.0,0.0
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,4-bit,0.0,0.0,0.0,0.0,0.0


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
9,Qwen/Qwen3-8B,gptq,Sinq,4-bit,1.552594,0.454422,1.248394,1.865661,1.718334
7,Qwen/Qwen3-8B,gptq,English,4-bit,1.586967,0.462869,1.274051,1.911259,1.736173
11,Qwen/Qwen3-8B,gptq,Tamil,4-bit,1.566731,0.445503,1.256366,1.906548,1.716401
8,Qwen/Qwen3-8B,gptq,Indonesian,4-bit,1.559996,0.457800,1.243969,1.877965,1.716841
10,Qwen/Qwen3-8B,gptq,Swahili,4-bit,1.565642,0.456699,1.272481,1.854443,1.724152
6,Qwen/Qwen3-8B,gptq,Chinese,4-bit,1.565742,0.465653,1.253915,1.894876,1.717118


...

## Shape
df text: (15, 19)
df std: (15, 19)
Display df


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-SINQ,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,39.595530,40.062125,39.947466,40.115798,40.419192,38.331823,39.595530,41.523714,NaN,44.224801,46.296442,38.107827,39.595530,40.119780,39.193255,40.222063,39.940935,37.424322
SW_Llama,30.056285,32.918649,35.268858,34.343137,33.018261,31.212913,34.342650,32.918649,36.902012,NaN,37.360703,32.886215,35.678429,32.918649,34.456392,33.305476,32.882074,30.807823,33.288619
ID_Llama,24.823783,25.845251,27.111507,27.035368,25.609484,26.443130,27.025912,25.845251,28.568594,NaN,26.986952,28.200670,27.777874,25.845251,27.187312,27.168395,25.944766,26.501118,27.072293
TA_Llama,2.311781,2.412915,2.503244,2.341632,2.454821,2.453562,2.436780,2.412915,2.695015,NaN,2.735899,2.699435,2.555277,2.412915,2.503889,2.345541,2.457776,2.460396,2.424910
EN_Llama,45.105214,43.628489,47.372066,46.727688,46.340421,46.679481,46.697379,43.628489,43.532462,NaN,44.632579,45.423450,44.928034,43.628489,45.535295,46.511978,46.849139,46.166003,45.672605
ZH_Aya,112.084228,122.265194,113.312037,112.280232,113.168497,112.105412,113.046247,122.265194,124.485867,NaN,122.105721,123.359929,117.615823,122.265194,120.301677,120.618816,119.716704,121.515856,116.573491
SW_Aya,347.245294,394.071986,401.770728,373.873691,365.351107,352.421007,381.202734,394.071986,397.648200,NaN,372.462067,337.157383,412.249550,394.071986,388.558360,389.291678,381.536862,364.872794,411.201310
ID_Aya,92.363250,98.200647,89.978133,90.784244,90.535852,89.279740,92.005672,98.200647,98.354503,NaN,93.870980,97.055222,98.401795,98.200647,96.781066,97.586160,95.089391,97.101196,96.739712
TA_Aya,3.798334,4.002676,3.947913,3.796851,3.977342,3.900542,3.929984,4.002676,4.021575,NaN,3.975950,3.920454,3.954161,4.002676,3.980362,3.848132,3.942369,3.988709,3.928528
EN_Aya,53.754913,55.876352,50.815876,51.355844,52.137953,50.556806,52.413353,55.876352,54.706773,NaN,55.686326,55.946115,55.432520,55.876352,55.022220,55.689060,55.559975,55.820594,55.483941


Display df std


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-SINQ,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,,,,,,,,,,,,,
SW_Llama,,,,,,,,,,,,,,,,,,,
ID_Llama,,,,,,,,,,,,,,,,,,,
TA_Llama,,,,,,,,,,,,,,,,,,,
EN_Llama,,,,,,,,,,,,,,,,,,,
ZH_Aya,,,,,,,,,,,,,,,,,,,
SW_Aya,,,,,,,,,,,,,,,,,,,
ID_Aya,,,,,,,,,,,,,,,,,,,
TA_Aya,,,,,,,,,,,,,,,,,,,
EN_Aya,,,,,,,,,,,,,,,,,,,


Writing to pdf ...


In [10]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            # {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"config.bit_width": f"{bit_width}-bit"},
                            # {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            # {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_{bit_width}bit_sinq_all_vert_final.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
31,Qwen/Qwen3-8B,gptq,3-bit,Sinq,53.160308,3.835802,27.044173,170.134416,75.035734
0,Qwen/Qwen3-8B,gptq,3-bit,English,40.188512,4.329966,24.304464,160.417519,62.901249
...,...,...,...,...,...,...,...,...,...
28,Qwen/Qwen3-14B,tacq,3-bit,English,33.208545,3.341025,18.200096,83.344899,48.417162
30,Qwen/Qwen3-14B,tacq,3-bit,Tamil,39.802250,2.560754,16.792030,74.304802,48.441249
29,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,34.557319,2.920792,15.548666,66.936384,49.449916
32,Qwen/Qwen3-14B,tacq,3-bit,Swahili,NaN,NaN,NaN,NaN,NaN


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
15,Qwen/Qwen3-8B,tacq,Sinq,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
13,Qwen/Qwen3-8B,tacq,English,3-bit,0.151631,0.039473,0.103480,2.745665,0.322409
17,Qwen/Qwen3-8B,tacq,Tamil,3-bit,0.787157,0.004207,0.143836,1.245594,0.341394
14,Qwen/Qwen3-8B,tacq,Indonesian,3-bit,0.473008,0.012477,0.037528,5.156465,0.841770
16,Qwen/Qwen3-8B,tacq,Swahili,3-bit,0.002507,0.024800,0.039048,0.484618,0.440133
12,Qwen/Qwen3-8B,tacq,Chinese,3-bit,0.376958,0.038915,0.161324,2.276247,0.174826
3,CohereLabs/aya-expanse-8b,tacq,Sinq,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
1,CohereLabs/aya-expanse-8b,tacq,English,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
5,CohereLabs/aya-expanse-8b,tacq,Tamil,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,3-bit,0.000000,0.000000,0.000000,0.000000,0.000000


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
15,Qwen/Qwen3-8B,gptq,Sinq,3-bit,1.725587,0.583856,1.432074,2.230792,1.875268
13,Qwen/Qwen3-8B,gptq,English,3-bit,1.607722,0.631171,1.395688,2.210527,1.804371
17,Qwen/Qwen3-8B,gptq,Tamil,3-bit,1.629727,0.466045,1.362567,2.151944,1.807234
14,Qwen/Qwen3-8B,gptq,Indonesian,3-bit,1.632738,0.578527,1.280456,2.135591,1.797347
16,Qwen/Qwen3-8B,gptq,Swahili,3-bit,1.644914,0.571095,1.347168,1.898244,1.813504
12,Qwen/Qwen3-8B,gptq,Chinese,3-bit,1.612060,0.622343,1.395127,2.209036,1.747992


...

## Shape
df text: (15, 19)
df std: (15, 19)
Display df


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-SINQ,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,50.598038,65.906926,57.598253,62.209873,62.640825,45.390110,50.598038,107.699684,NaN,68.901162,493.790219,44.496971,50.598038,57.113641,48.481814,52.050964,56.756861,41.277322
SW_Llama,30.056285,54.643300,76.299433,63.393429,54.521151,36.941053,70.289028,54.643300,140.046129,NaN,64.677236,127.097711,79.511096,54.643300,56.283017,46.878139,43.463953,34.765476,50.868524
ID_Llama,24.823783,36.127868,49.034587,42.820579,32.114254,38.430521,45.473741,36.127868,80.624982,NaN,33.437626,237.449086,52.094113,36.127868,39.772587,35.062178,28.618159,34.721147,35.462255
TA_Llama,2.311781,3.214247,4.619366,2.477804,3.813212,3.841755,3.669907,3.214247,6.090120,NaN,4.954544,8.438959,3.885786,3.214247,4.269030,2.430107,3.463090,3.554056,3.123373
EN_Llama,45.105214,50.553287,54.002999,56.975352,56.643278,55.005956,56.127699,50.553287,51.661761,NaN,52.608449,82.705680,55.081591,50.553287,48.270659,50.732388,50.051469,51.241933,50.027819
ZH_Aya,112.084228,134.500910,170.108220,262.474523,151.644070,153.626228,124.030659,134.500910,153.807992,NaN,148.348093,152.570887,127.689433,134.500910,132.582898,135.092829,132.251605,139.732992,119.321911
SW_Aya,347.245294,598.420257,701.944022,866.680681,601.327532,386.510893,628.567549,598.420257,694.409390,NaN,581.597703,372.344141,698.011042,598.420257,488.415269,494.785107,505.833186,351.026762,561.057641
ID_Aya,92.363250,108.035480,122.900772,170.286285,100.952621,109.504752,108.384593,108.035480,128.241772,NaN,106.431040,117.153934,123.701714,108.035480,104.218850,107.600928,96.903556,105.485363,106.001720
TA_Aya,3.798334,4.628170,5.489648,3.942703,4.941318,4.902821,5.048185,4.628170,5.190608,NaN,4.846758,4.707457,4.957356,4.628170,4.598233,3.914037,4.432267,4.394636,4.441016
EN_Aya,53.754913,60.296624,55.716955,59.996131,58.331153,56.882036,56.490654,60.296624,70.334848,NaN,70.159288,71.447728,71.276244,60.296624,56.603639,59.778306,57.988980,59.710328,58.374838


Display df std


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-SINQ,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,,,,,,,,,,,,,
SW_Llama,,,,,,,,,,,,,,,,,,,
ID_Llama,,,,,,,,,,,,,,,,,,,
TA_Llama,,,,,,,,,,,,,,,,,,,
EN_Llama,,,,,,,,,,,,,,,,,,,
ZH_Aya,,,,,,,,,,,,,,,,,,,
SW_Aya,,,,,,,,,,,,,,,,,,,
ID_Aya,,,,,,,,,,,,,,,,,,,
TA_Aya,,,,,,,,,,,,,,,,,,,
EN_Aya,,,,,,,,,,,,,,,,,,,


Writing to pdf ...


In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            # {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"config.bit_width": f"{bit_width}-bit"},
                            # {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": base_model},
                            # {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": base_model},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_{bit_width}bit_sinq_all_vert_final.pdf")

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,1.669901e+01,6.453465e+01,4.698081e+01
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,9.236325e+01,3.472453e+02,1.120842e+02
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,2.482378e+01,3.005629e+01,3.682178e+01
41,Qwen/Qwen3-8B,gptq,2-bit,Sinq,778028.489669,221688.724629,4.557271e+06,4.380327e+06,2.175599e+06
0,Qwen/Qwen3-8B,gptq,2-bit,English,13062.876244,48256.105485,3.985446e+04,3.823152e+04,3.428624e+04
...,...,...,...,...,...,...,...,...,...
38,Qwen/Qwen3-14B,tacq,2-bit,English,42.305540,17.766269,1.094552e+02,3.921057e+02,1.040427e+02
40,Qwen/Qwen3-14B,tacq,2-bit,Tamil,127.865788,2.734088,4.718777e+01,3.796860e+02,8.750628e+01
39,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,55.124529,8.474698,1.805037e+01,2.543852e+02,8.970690e+01
52,Qwen/Qwen3-14B,tacq,2-bit,Swahili,NaN,NaN,NaN,NaN,NaN


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
15,Qwen/Qwen3-8B,tacq,Sinq,2-bit,0.000000,0.000000,0.000000,0.000000,0.000000
13,Qwen/Qwen3-8B,tacq,English,2-bit,0.323566,0.665306,3.515737,10.556345,3.317178
17,Qwen/Qwen3-8B,tacq,Tamil,2-bit,0.461728,0.010456,14.028029,144.154089,2.312004
14,Qwen/Qwen3-8B,tacq,Indonesian,2-bit,0.534628,0.818386,0.167729,23.810479,4.750264
16,Qwen/Qwen3-8B,tacq,Swahili,2-bit,0.458314,0.499177,0.602931,1.532377,10.103402
12,Qwen/Qwen3-8B,tacq,Chinese,2-bit,0.907797,1.544423,5.781454,15.010444,0.752618
3,CohereLabs/aya-expanse-8b,tacq,Sinq,2-bit,0.000000,0.000000,0.000000,0.000000,0.000000
1,CohereLabs/aya-expanse-8b,tacq,English,2-bit,0.218186,1.433973,4.411970,233.931693,12.841869
5,CohereLabs/aya-expanse-8b,tacq,Tamil,2-bit,1.176844,0.012800,9.131215,52.022620,5.145499
2,CohereLabs/aya-expanse-8b,tacq,Indonesian,2-bit,0.472105,0.360203,2.482059,161.906780,15.790946


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
15,Qwen/Qwen3-8B,gptq,Sinq,2-bit,5.890996,5.345744,6.658705,6.641507,6.337579
13,Qwen/Qwen3-8B,gptq,English,2-bit,4.108156,4.662199,4.593693,4.606363,4.596503
17,Qwen/Qwen3-8B,gptq,Tamil,2-bit,4.642051,0.873260,4.664672,4.670482,4.531575
14,Qwen/Qwen3-8B,gptq,Indonesian,2-bit,4.544935,4.508749,3.226292,4.541529,4.624799
16,Qwen/Qwen3-8B,gptq,Swahili,2-bit,4.643958,4.572244,4.622783,3.166420,4.405375
12,Qwen/Qwen3-8B,gptq,Chinese,2-bit,4.489204,4.365599,4.580982,4.616916,4.249957


...

## Shape
df text: (15, 19)
df std: (15, 19)
Display df


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-SINQ,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,3.221260e+04,52937.025188,31079.064005,46846.875085,58306.035056,3191.528221,3.221260e+04,6.502951e+06,NaN,1.828967e+06,2.602418e+06,3788.615196,3.221260e+04,831.292973,245.952841,374.675809,872.837297,65.893885
SW_Llama,30.056285,8.899394e+04,38619.420435,54418.192756,21461.796460,2589.449002,49507.095252,8.899394e+04,3.413639e+04,NaN,1.968422e+04,5.162781e+04,35005.098924,8.899394e+04,560.461792,284.107680,213.018242,44.153591,438.002381
ID_Llama,24.823783,7.777345e+04,41286.884848,45163.124564,2450.187930,24721.264360,50058.363746,7.777345e+04,5.692653e+04,NaN,4.410187e+04,9.081436e+04,63450.902216,7.777345e+04,558.862629,244.421507,48.206539,151.038203,411.683272
TA_Llama,2.311781,4.711932e+04,462570.732187,10.981143,128857.950967,402166.581074,58897.957168,4.711932e+04,8.326694e+06,NaN,2.886942e+06,1.401443e+06,132852.344038,4.711932e+04,70.142513,2.766078,14.534124,16.163262,8.765659
EN_Llama,45.105214,8.261823e+03,1283.204545,8678.450527,6712.454060,8558.666771,7777.211806,8.261823e+03,9.628502e+02,NaN,1.449435e+04,3.808274e+04,13740.636867,8.261823e+03,74.420949,113.737093,99.134160,112.516157,106.758411
ZH_Aya,112.084228,2.834583e+03,19168.145370,9278.960985,9950.510277,14767.553330,1029.857317,2.834583e+03,7.409755e+06,NaN,1.631137e+06,1.004871e+06,274.084188,2.834583e+03,360.637008,275.617452,292.760948,406.131890,135.838040
SW_Aya,347.245294,1.150134e+04,22225.967596,21938.558958,14221.136656,1591.607966,20855.464249,1.150134e+04,9.978332e+03,NaN,5.561333e+03,4.189173e+02,50822.574116,1.150134e+04,1588.158444,1544.971890,1344.702942,332.042594,2235.354059
ID_Aya,92.363250,1.937413e+03,6090.955316,6473.671674,545.811702,4321.037729,6165.939366,1.937413e+03,4.766603e+04,NaN,2.328256e+02,1.044489e+03,484718.872812,1.937413e+03,306.205275,225.322198,117.137641,202.212119,297.658705
TA_Aya,3.798334,8.555107e+01,22325.499442,6.436282,4200.197925,1501.481445,4226.914393,8.555107e+01,4.110211e+05,NaN,4.490107e+05,6.071352e+05,290134.141033,8.555107e+01,14.739504,4.049754,12.200776,10.231536,12.421246
EN_Aya,53.754913,3.607169e+02,241.836277,745.679240,522.582423,824.976137,1006.574886,3.607169e+02,1.366964e+02,NaN,2.688014e+02,3.345557e+02,4622.001841,3.607169e+02,69.141747,93.205527,80.995250,97.275513,85.909020


Display df std


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,SliM-LLM-SINQ,SliM-LLM-EN,SliM-LLM-TA,SliM-LLM-ID,SliM-LLM-SW,SliM-LLM-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,1.4e+4,642.8,1.0e+4,1.2e+4,56.51,,5.3e+6,,6.1e+4,1.3e+6,1.4e+3,,133.0,30.47,7.06,284.0,2.90
SW_Llama,,,4.4e+3,8.1e+3,2.0e+3,1.5e+3,6.6e+3,,1.4e+4,,1.4e+4,1.6e+4,1.5e+4,,137.7,45.49,4.65,0.98,31.15
ID_Llama,,,9.3e+3,6.3e+3,317.8,2.4e+3,1.4e+4,,3.3e+4,,4.0e+4,3.5e+4,8.8e+3,,149.1,10.45,0.48,15.13,69.98
TA_Llama,,,,1.46,,,1.5e+3,,4.9e+6,,9.4e+4,8.1e+5,8.0e+4,,28.76,0.02,3.24,0.27,0.01
EN_Llama,,,17.19,201.3,589.3,1.9e+3,97.15,,347.5,,1.0e+4,8.9e+3,6.7e+3,,3.12,9.28,4.39,4.12,4.08
ZH_Aya,,,1.5e+3,1.3e+3,866.1,1.2e+3,18.95,,1.2e+6,,4.2e+5,6.5e+5,13.88,,12.84,5.15,15.79,4.07,0.39
SW_Aya,,,1.3e+3,2.1e+3,1.2e+3,33.02,2.3e+3,,2.1e+3,,1.1e+3,37.52,2.6e+4,,233.9,52.02,161.9,8.99,306.7
ID_Aya,,,785.6,453.4,36.47,598.0,748.5,,1.6e+4,,10.86,407.5,4.7e+4,,4.41,9.13,2.48,2.53,17.78
TA_Aya,,,6.7e+3,0.24,1.0e+3,576.6,1.1e+3,,2.8e+5,,3.2e+5,8.3e+5,2.7e+5,,1.43,0.01,0.36,0.19,0.98
EN_Aya,,,6.35,60.78,25.46,80.77,75.66,,8.65,,22.66,117.2,3.5e+3,,0.22,1.18,0.47,1.53,1.51


Writing to pdf ...


###### Qwen3-14B

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"config.bit_width": f"{bit_width}-bit"},
                            # {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-14B"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_14b_{bit_width}bit_sinq_all_vert_final.pdf")

/tmp/ipykernel_5078/1242997687.py:187: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-14B,Unquantized,32-bit,Unquantized,31.906740,2.528679,15.144081,46.729738,44.586565
6,Qwen/Qwen3-14B,gptq,3-bit,Sinq,65.452441,3.086842,22.252021,140.257758,58.400786
0,Qwen/Qwen3-14B,gptq,3-bit,English,36.570983,3.434160,19.298548,106.342203,54.280184
1,Qwen/Qwen3-14B,gptq,3-bit,English,NaN,NaN,NaN,NaN,NaN
2,Qwen/Qwen3-14B,gptq,3-bit,English,NaN,NaN,NaN,NaN,NaN
7,Qwen/Qwen3-14B,gptq,3-bit,Tamil,39.736187,2.653994,18.600899,101.042983,53.330655
3,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,38.121960,3.113559,16.842957,87.253248,54.865941
4,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,NaN,NaN,NaN,NaN,NaN
5,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,NaN,NaN,NaN,NaN,NaN
8,Qwen/Qwen3-14B,gptq,3-bit,Swahili,37.600310,3.034574,18.033382,53.112277,54.543040


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
3,Qwen/Qwen3-14B,tacq,Sinq,3-bit,0.0,0.0,0.0,0.0,0.0
1,Qwen/Qwen3-14B,tacq,English,3-bit,0.0,0.0,0.0,0.0,0.0
5,Qwen/Qwen3-14B,tacq,Tamil,3-bit,0.0,0.0,0.0,0.0,0.0
2,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.0,0.0,0.0,0.0,0.0
4,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.0,0.0,0.0,0.0,0.0
0,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.0,0.0,0.0,0.0,0.0


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-14B


...

## Shape
df text: (5, 13)
df std: (5, 13)
Display df


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Qwen3 14B,44.586565,58.400786,54.280184,53.330655,54.865941,54.543040,49.925631,58.400786,48.417162,48.441249,49.449916,49.457979,45.356019
SW_Qwen3 14B,46.729738,140.257758,106.342203,101.042983,87.253248,53.112277,102.708718,140.257758,83.344899,74.304802,66.936384,46.640123,81.958071
ID_Qwen3 14B,15.144081,22.252021,19.298548,18.600899,16.842957,18.033382,19.419783,22.252021,18.200096,16.792030,15.548666,16.301946,17.858717
TA_Qwen3 14B,2.528679,3.086842,3.434160,2.653994,3.113559,3.034574,3.251492,3.086842,3.341025,2.560754,2.920792,2.839776,3.098871
EN_Qwen3 14B,31.906740,65.452441,36.570983,39.736187,38.121960,37.600310,37.183523,65.452441,33.208545,39.802250,34.557319,34.834344,34.466696


Display df std


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Qwen3 14B,,,,,,,,,,,,,
SW_Qwen3 14B,,,,,,,,,,,,,
ID_Qwen3 14B,,,,,,,,,,,,,
TA_Qwen3 14B,,,,,,,,,,,,,
EN_Qwen3 14B,,,,,,,,,,,,,


Writing to pdf ...


In [15]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"config.bit_width": f"{bit_width}-bit"},
                            # {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            # {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$gt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-14B"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_14b_{bit_width}bit_sinq_all_vert_final.pdf")

/tmp/ipykernel_5078/1242997687.py:187: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-14B,Unquantized,32-bit,Unquantized,31.906740,2.528679,15.144081,46.729738,44.586565
6,Qwen/Qwen3-14B,gptq,2-bit,Sinq,93245.028692,3703.413056,21716.156280,189746.386430,134401.759076
0,Qwen/Qwen3-14B,gptq,2-bit,English,NaN,NaN,NaN,NaN,NaN
1,Qwen/Qwen3-14B,gptq,2-bit,English,NaN,NaN,NaN,NaN,NaN
7,Qwen/Qwen3-14B,gptq,2-bit,English,1731.421552,126724.181707,15134.717731,63182.296317,98882.102925
4,Qwen/Qwen3-14B,gptq,2-bit,Tamil,NaN,NaN,NaN,NaN,NaN
5,Qwen/Qwen3-14B,gptq,2-bit,Tamil,NaN,NaN,NaN,NaN,NaN
9,Qwen/Qwen3-14B,gptq,2-bit,Tamil,58067.513363,4.540477,8132.349136,16367.378790,126329.283396
2,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,NaN,NaN,NaN,NaN,NaN
3,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,NaN,NaN,NaN,NaN,NaN


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
3,Qwen/Qwen3-14B,tacq,Sinq,2-bit,0.0,0.0,0.0,0.0,0.0
1,Qwen/Qwen3-14B,tacq,English,2-bit,0.0,0.0,0.0,0.0,0.0
5,Qwen/Qwen3-14B,tacq,Tamil,2-bit,0.0,0.0,0.0,0.0,0.0
2,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.0,0.0,0.0,0.0,0.0
4,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.0,0.0,0.0,0.0,0.0
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.0,0.0,0.0,0.0,0.0


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-14B


...

## Shape
df text: (5, 13)
df std: (5, 13)
Display df


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Qwen3 14B,44.586565,134401.759076,98882.102925,126329.283396,59876.095003,187673.707091,2048.140191,134401.759076,104.042738,87.506276,89.706897,113.418869,53.193292
SW_Qwen3 14B,46.729738,189746.386430,63182.296317,16367.378790,5962.964127,306.922576,57565.977822,189746.386430,392.105679,379.686037,254.385204,49.380839,438.827932
ID_Qwen3 14B,15.144081,21716.156280,15134.717731,8132.349136,105.458845,6430.714809,15931.801910,21716.156280,109.455182,47.187771,18.050369,30.467371,100.802627
TA_Qwen3 14B,2.528679,3703.413056,126724.181707,4.540477,509.449465,348.604903,13161.527566,3703.413056,17.766269,2.734088,8.474698,9.076365,19.457560
EN_Qwen3 14B,31.906740,93245.028692,1731.421552,58067.513363,9109.144965,40439.801741,7391.931214,93245.028692,42.305540,127.865788,55.124529,65.045889,56.342883


Display df std


,UQ,GPTQ-SINQ,GPTQ-EN,GPTQ-TA,GPTQ-ID,GPTQ-SW,GPTQ-ZH,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Qwen3 14B,,,,,,,,,,,,,
SW_Qwen3 14B,,,,,,,,,,,,,
ID_Qwen3 14B,,,,,,,,,,,,,
TA_Qwen3 14B,,,,,,,,,,,,,
EN_Qwen3 14B,,,,,,,,,,,,,


Writing to pdf ...


##### All data - Each Quant

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [4]:
  for quantization_technique in ["gptq", "slimllm", "tacq"]:
    runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                    filters={"$or": [
                            { "$and": [
                              {"config.quantization_technique": quantization_technique.lower()},
                              {"config.bit_width": f"{bit_width}-bit"},
                              {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              # {"created_at": {"$lt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                            ]
                            },
                            { "$and": [
                              {"config.quantization_technique": "sinq"},
                              {"config.bit_width": f"{bit_width}-bit"},
                              # {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              # {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              # {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$gt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            ]
                            },
                            { "$and" : [
                                {"config.quantization_technique": "Unquantized"},
                                # {"config.evaluation_dataset": evaluation_dataset},
                                {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                                # {"config.base_model": base_model},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                                {"config.enable_thinking": False},
                                {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                            ]}
                        ]}
                    )

    eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
    # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
    # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
    # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
    # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
    # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

    # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
    perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_{bit_width}bit_{quantization_technique}_all_vert_final.pdf")

/tmp/ipykernel_3071/1242997687.py:187: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
15,Qwen/Qwen3-8B,gptq,4-bit,Sinq,35.693894,2.847228,17.717172,73.394102,52.279770
0,Qwen/Qwen3-8B,gptq,4-bit,English,38.633773,2.903144,18.795383,81.519027,54.471988
2,Qwen/Qwen3-8B,gptq,4-bit,Tamil,36.874922,2.789352,18.045387,80.639509,52.047636
1,Qwen/Qwen3-8B,gptq,4-bit,Indonesian,36.307482,2.869462,17.537566,75.503067,52.100443
3,Qwen/Qwen3-8B,gptq,4-bit,Swahili,36.782606,2.862192,18.727565,71.522493,52.984904
4,Qwen/Qwen3-8B,gptq,4-bit,Chinese,36.791005,2.921814,17.943838,78.501095,52.133661
16,CohereLabs/aya-expanse-8b,gptq,4-bit,Sinq,55.876352,4.002676,98.200647,394.071986,122.265194


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


...

## Shape
df text: (15, 7)
df std: (15, 7)
Display df


,UQ,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,39.595530,40.119780,39.193255,40.222063,39.940935,37.424322
SW_Llama,30.056285,32.918649,34.456392,33.305476,32.882074,30.807823,33.288619
ID_Llama,24.823783,25.845251,27.187312,27.168395,25.944766,26.501118,27.072293
TA_Llama,2.311781,2.412915,2.503889,2.345541,2.457776,2.460396,2.424910
EN_Llama,45.105214,43.628489,45.535295,46.511978,46.849139,46.166003,45.672605
ZH_Aya,112.084228,122.265194,120.301677,120.618816,119.716704,121.515856,116.573491
SW_Aya,347.245294,394.071986,388.558360,389.291678,381.536862,364.872794,411.201310
ID_Aya,92.363250,98.200647,96.781066,97.586160,95.089391,97.101196,96.739712
TA_Aya,3.798334,4.002676,3.980362,3.848132,3.942369,3.988709,3.928528
EN_Aya,53.754913,55.876352,55.022220,55.689060,55.559975,55.820594,55.483941


Display df std


,UQ,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,
SW_Llama,,,,,,,
ID_Llama,,,,,,,
TA_Llama,,,,,,,
EN_Llama,,,,,,,
ZH_Aya,,,,,,,
SW_Aya,,,,,,,
ID_Aya,,,,,,,
TA_Aya,,,,,,,
EN_Aya,,,,,,,


Writing to pdf ...


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  for quantization_technique in ["gptq", "slimllm", "tacq"]:
    runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                    filters={"$or": [
                            { "$and": [
                              {"config.quantization_technique": quantization_technique.lower()},
                              {"config.bit_width": f"{bit_width}-bit"},
                              {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              # {"created_at": {"$lt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                            ]
                            },
                            { "$and": [
                              {"config.quantization_technique": "sinq"},
                              {"config.bit_width": f"{bit_width}-bit"},
                              # {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              # {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              # {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$gt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            ]
                            },
                            { "$and" : [
                                {"config.quantization_technique": "Unquantized"},
                                # {"config.evaluation_dataset": evaluation_dataset},
                                {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                                # {"config.base_model": base_model},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                                {"config.enable_thinking": False},
                                {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                            ]}
                        ]}
                    )

    eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
    # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
    # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
    # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
    # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
    # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

    # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
    perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_{bit_width}bit_{quantization_technique}_all_vert_final.pdf")

/tmp/ipykernel_3071/1242997687.py:187: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,16.699008,64.534645,46.980806
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,92.363250,347.245294,112.084228
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,24.823783,30.056285,36.821779
31,Qwen/Qwen3-8B,gptq,3-bit,Sinq,53.160308,3.835802,27.044173,170.134416,75.035734
0,Qwen/Qwen3-8B,gptq,3-bit,English,40.188512,4.329966,24.304464,160.417519,62.901249
15,Qwen/Qwen3-8B,gptq,3-bit,English,40.949766,4.233635,25.688953,160.373092,64.743586
16,Qwen/Qwen3-8B,gptq,3-bit,English,40.436502,4.268332,24.618706,166.343463,63.556982
2,Qwen/Qwen3-8B,gptq,3-bit,Tamil,42.012760,2.919523,22.620578,144.263196,62.475377
19,Qwen/Qwen3-8B,gptq,3-bit,Tamil,42.729569,2.937405,23.351523,141.058965,65.202294
20,Qwen/Qwen3-8B,gptq,3-bit,Tamil,43.151184,2.916437,23.161336,140.340460,64.788691


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


...

## Shape
df text: (15, 7)
df std: (15, 7)
Display df


,UQ,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,50.598038,57.113641,48.481814,52.050964,56.756861,41.277322
SW_Llama,30.056285,54.643300,56.283017,46.878139,43.463953,34.765476,50.868524
ID_Llama,24.823783,36.127868,39.772587,35.062178,28.618159,34.721147,35.462255
TA_Llama,2.311781,3.214247,4.269030,2.430107,3.463090,3.554056,3.123373
EN_Llama,45.105214,50.553287,48.270659,50.732388,50.051469,51.241933,50.027819
ZH_Aya,112.084228,134.500910,132.582898,135.092829,132.251605,139.732992,119.321911
SW_Aya,347.245294,598.420257,488.415269,494.785107,505.833186,351.026762,561.057641
ID_Aya,92.363250,108.035480,104.218850,107.600928,96.903556,105.485363,106.001720
TA_Aya,3.798334,4.628170,4.598233,3.914037,4.432267,4.394636,4.441016
EN_Aya,53.754913,60.296624,56.603639,59.778306,57.988980,59.710328,58.374838


Display df std


,UQ,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,,,,,
SW_Llama,,,,,,,
ID_Llama,,,,,,,
TA_Llama,,,,,,,
EN_Llama,,,,,,,
ZH_Aya,,,,,,,
SW_Aya,,,,,,,
ID_Aya,,,,,,,
TA_Aya,,,,,,,
EN_Aya,,,,,,,


Writing to pdf ...


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  for quantization_technique in ["gptq", "slimllm", "tacq"]:
    runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                    filters={"$or": [
                            { "$and": [
                              {"config.quantization_technique": quantization_technique.lower()},
                              {"config.bit_width": f"{bit_width}-bit"},
                              {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              # {"created_at": {"$lt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              # {"$not": {"$and": [{"config.quantization_technique": "gptq"}, {"config.calibration_language": "Swahili"}]}}
                            ]
                            },
                            { "$and": [
                              {"config.quantization_technique": "sinq"},
                              {"config.bit_width": f"{bit_width}-bit"},
                              # {"config.num_calibration_samples": 128},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"$or": [
                                {"config.evaluation_dataset": f"floresplus-evaluate"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                                {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                              ]},
                              # {"config.base_model": base_model},
                              # {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                              # {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$gt": '2026-07-01'}},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                            ]
                            },
                            { "$and" : [
                                {"config.quantization_technique": "Unquantized"},
                                # {"config.evaluation_dataset": evaluation_dataset},
                                {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                                # {"config.base_model": base_model},
                              # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                              # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                                {"config.enable_thinking": False},
                                {"config.output_type": "perplexity"},
                              {"created_at": {"$gt": '2026-01-15'}},
                              {"created_at": {"$lt": '2026-07-01'}},
                            ]}
                        ]}
                    )

    eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
    # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
    # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
    # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
    # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
    # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

    # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
    perplexity_heatmap_deliver_vertical_sinq(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=True, two_color=False, display_colorbar=False, embed_sinq=True, write_pdf=f"ablation_floresplus_{bit_width}bit_{quantization_technique}_all_vert_final.pdf")

/tmp/ipykernel_3071/1242997687.py:187: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.428698,2.741605,1.669901e+01,6.453465e+01,4.698081e+01
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.754913,3.798334,9.236325e+01,3.472453e+02,1.120842e+02
2,meta-llama/Llama-3.1-8B-Instruct,Unquantized,32-bit,Unquantized,45.105214,2.311781,2.482378e+01,3.005629e+01,3.682178e+01
41,Qwen/Qwen3-8B,gptq,2-bit,Sinq,778028.489669,221688.724629,4.557271e+06,4.380327e+06,2.175599e+06
0,Qwen/Qwen3-8B,gptq,2-bit,English,13062.876244,48256.105485,3.985446e+04,3.823152e+04,3.428624e+04
...,...,...,...,...,...,...,...,...,...
37,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,NaN,NaN,NaN,NaN,NaN
38,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,NaN,NaN,NaN,NaN,NaN
46,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,9109.144965,509.449465,1.054588e+02,5.962964e+03,5.987610e+04
58,Qwen/Qwen3-14B,gptq,2-bit,Swahili,NaN,NaN,NaN,NaN,NaN


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans


Index(['Base Model', 'Quantization Technique', 'Calibration Language',
       'Bit Width', 'eng_Latn', 'tam_Taml', 'ind_Latn', 'swh_Latn',
       'cmn_Hans'],
      dtype='object')

start model: Qwen/Qwen3-8B


...

## Shape
df text: (15, 7)
df std: (15, 7)
Display df


,UQ,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,36.821779,3.221260e+04,831.292973,245.952841,374.675809,872.837297,65.893885
SW_Llama,30.056285,8.899394e+04,560.461792,284.107680,213.018242,44.153591,438.002381
ID_Llama,24.823783,7.777345e+04,558.862629,244.421507,48.206539,151.038203,411.683272
TA_Llama,2.311781,4.711932e+04,70.142513,2.766078,14.534124,16.163262,8.765659
EN_Llama,45.105214,8.261823e+03,74.420949,113.737093,99.134160,112.516157,106.758411
ZH_Aya,112.084228,2.834583e+03,360.637008,275.617452,292.760948,406.131890,135.838040
SW_Aya,347.245294,1.150134e+04,1588.158444,1544.971890,1344.702942,332.042594,2235.354059
ID_Aya,92.363250,1.937413e+03,306.205275,225.322198,117.137641,202.212119,297.658705
TA_Aya,3.798334,8.555107e+01,14.739504,4.049754,12.200776,10.231536,12.421246
EN_Aya,53.754913,3.607169e+02,69.141747,93.205527,80.995250,97.275513,85.909020


Display df std


,UQ,TaCQ-SINQ,TaCQ-EN,TaCQ-TA,TaCQ-ID,TaCQ-SW,TaCQ-ZH
ZH_Llama,,,133.0,30.47,7.06,284.0,2.90
SW_Llama,,,137.7,45.49,4.65,0.98,31.15
ID_Llama,,,149.1,10.45,0.48,15.13,69.98
TA_Llama,,,28.76,0.02,3.24,0.27,0.01
EN_Llama,,,3.12,9.28,4.39,4.12,4.08
ZH_Aya,,,12.84,5.15,15.79,4.07,0.39
SW_Aya,,,233.9,52.02,161.9,8.99,306.7
ID_Aya,,,4.41,9.13,2.48,2.53,17.78
TA_Aya,,,1.43,0.01,0.36,0.19,0.98
EN_Aya,,,0.22,1.18,0.47,1.53,1.51


Writing to pdf ...


##### Stats: aggresive

In [ ]:
def perplexity_aggresive_target_not(eval_languages, runs, base_model, quantization_technique, bit_width, eval_dataset, custom_title="", filter_result='', hf_evaluate=False, compare_std=False, gmean_aggregate=False):
  language_sort = {
      # Flores Plus
      "eng_Latn": 0, "fra_Latn": 1, "hin_Deva": 2, "npi_Deva": 3, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "cmn_Hans": 12, "jpn_Jpan": 13, "kor_Hang": 14, "arb_Arab" : 15,\
      # Wikipedia
      "en": 0, "fr": 1, "hi": 2, "ne": 3, "ta": 6, "te": 7, "id": 8, "ms": 9, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  }
  flores_plus_add = ["afr_Latn", "dan_Latn", "deu_Latn", "eng_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 0 + index / 100
  flores_plus_add = ["kan_Knda", "mal_Mlym", "tam_Taml", "tel_Telu"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 6 + index / 100
  #flores_plus_add = ["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "ind_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"]
  flores_plus_add = ["ind_Latn", "zsm_Latn", "bjn_Latn", "min_Latn", "ace_Latn", "ban_Latn", "bug_Latn", "jav_Latn", "sun_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "fil_Latn", "war_Latn"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 8 + index / 100
  flores_plus_add = ["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "swh_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 10 + index / 100
  flores_plus_add = ["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant", "cmn_Hans"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 12 + index / 100
  eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))

  calib_mapper = {
      "English": "eng_Latn" if eval_dataset.lower() == "floresplus" else "en",
      "Indonesian": "ind_Latn" if eval_dataset.lower() == "floresplus" else "id",
      "Tamil": "tam_Taml" if eval_dataset.lower() == "floresplus" else "ta",
      "Swahili": "swh_Latn" if eval_dataset.lower() == "floresplus" else "sw",
      "Chinese": "cmn_Hans" if eval_dataset.lower() == "floresplus" else "zh",
  }
  reverse_calib_mapper = {value: key for key, value in calib_mapper.items()}

  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language"]
  columns.extend(eval_languages)
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 '32-bit': 0, '2-bit': 1, '3-bit': 2, '4-bit': 3, '8-bit': 4}

  data_acc = []
  data_nancount = []

  unquantized_data_acc = []


  # Iterate over the runs and print details
  for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue

      run_data_acc = []
      run_data_nancount = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      run_data_nancount.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      if not hf_evaluate:
        for lang in eval_languages:
          if lang in ["zh", "Chinese", "cmn_Hans", "ja", "Japanese", "jpn_Jpan"]:
            acc = run.summary.get(f"{eval_dataset.lower()}_byte_acc_{lang}", None)
            nancount = run.summary.get(f"{eval_dataset.lower()}_count_nan_count_{lang}", 0)
            if acc:
              # acc = math.pow(acc, 1/3)
              pass
            run_data_acc.append(acc)
            run_data_nancount.append(nancount)
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None))
            run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nan_count_{lang}", 0))
      elif hf_evaluate:
        for lang in eval_languages:
          if gmean_aggregate:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_gmean_perplexity_{lang}", None))
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_token_perplexity_{lang}", None))
          run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nancount_{lang}", 0))

      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
        data_nancount.append(run_data_nancount)
      else:
        data_acc.append(run_data_acc)
        data_nancount.append(run_data_nancount)


  import numpy as np
  df_unquantized_acc = pd.DataFrame(unquantized_data_acc, columns=columns)
  df_unquantized_acc_chief_std = df_unquantized_acc.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std")

  df_acc = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Quantization Technique', 'Calibration Language', 'Bit Width'], key=lambda x: x.map(custom_sort))
  df_acc_std = df_acc.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_acc = df_acc.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  df_acc[df_acc.columns[4:]] = df_acc[df_acc.columns[4:]].astype(float)
  df_unquantized_acc[df_unquantized_acc.columns[4:]] = df_unquantized_acc[df_unquantized_acc.columns[4:]].astype(float)

  pd.set_option('display.float_format','{:.1f}'.format)
  # display(df_unquantized_acc)

  # # Perplexity related, change order of PPL
  # df_acc.iloc[:, 5:] = df_acc.iloc[:, 5:].apply(np.log10)
  # df_unquantized_acc.iloc[:, 5:] = df_unquantized_acc.iloc[:, 5:].apply(np.log10)

  calibration_languages = df_acc["Calibration Language"].unique()
  quants = df_acc["Quantization Technique"].unique()
  base_models = df_acc["Base Model"].unique()
  bits = df_acc["Bit Width"].unique()

  flag = True
  df = pd.DataFrame(data=[], columns=["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Target", "Mean Non Target",
                                      "Direct over Unquantized", "Cross over Unquantized", "Direct over Default", "Cross over Default", "Direct over Cross", "Cross over Direct"])
  for quant in quants:
    for bit in bits:
      for base_model in base_models:
        df_filter_uq = df_unquantized_acc[df_unquantized_acc["Base Model"] == base_model]
        df_filter = df_acc_std.loc[df_acc_std["Quantization Technique"] == quant][df_acc_std["Bit Width"] == bit][df_acc_std["Base Model"] == base_model] if compare_std else df_acc.loc[df_acc["Quantization Technique"] == quant][df_acc["Bit Width"] == bit][df_acc["Base Model"] == base_model]
        mean_ppl = df_filter.iloc[:, 4:].prod(axis=0)
        default_ppl = df_filter[df_filter["Calibration Language"] == "English"].iloc[0, 4:]
        count_ppl = df_filter.iloc[:, 4:].count(axis=0)
        df_filter = df_filter.reset_index(drop=True)
        if (quant, base_model) == ("slimllm", "CohereLabs/aya-expanse-8b"):
          mean_ppl = mean_ppl.drop(calib_mapper["Tamil"], axis=0)
          default_ppl = default_ppl.drop(calib_mapper["Tamil"], axis=0)
          count_ppl = count_ppl.drop(calib_mapper["Tamil"], axis=0)
          df_filter_uq = df_filter_uq.drop(calib_mapper["Tamil"], axis=1)

        df_filter.loc[:, "Target"] = pd.Series([df_filter.loc[i, calib_mapper[calib]] for i, calib in enumerate(df_filter["Calibration Language"].unique())])
        df_filter.loc[:, "Mean Non Target"] = (mean_ppl.values / df_filter["Target"].values) ** (1/(count_ppl.values - 1))
        # df_filter.loc[:, "Power Ratio"] = (10 ** df_filter["Mean Non Target"].values / 10 ** df_filter["Target"].values)

        # [0, x:] index is transforming from 2D (df) to 1D array (series)
        # This will broke if Unquantized have more than 1 instance
        df_filter.loc[:, "Direct over Unquantized"] = (df_filter["Target"].values / df_filter_uq.iloc[0, 4:].values) * 100
        df_filter.loc[:, "Cross over Unquantized"] = (df_filter["Mean Non Target"].values / df_filter_uq.iloc[0, 4:].values) * 100
        df_filter.loc[:, "Direct over Default"] = ((default_ppl.values - df_filter["Target"].values) / default_ppl.values) *100
        df_filter.loc[:, "Cross over Default"] = ((default_ppl.values - df_filter["Mean Non Target"].values) / default_ppl.values) * 100
        # df_filter.loc[:, "Direct over Cross"] = (df_filter["Target"].values / df_filter["Mean Non Target"].values) * 100
        df_filter.loc[:, "Direct over Cross"] = (df_filter["Target"].values / df_filter["Mean Non Target"].values)
        df_filter.loc[:, "Cross over Direct"] = (df_filter["Mean Non Target"].values / df_filter["Target"].values)
        # df_filter.drop(df_filter.columns[5:].drop(["Target", "Mean Non Target"]), axis=1)
        # if flag:
        #   print(quant, bit, base_model)
        #   display(df_filter["Target"])
        #   display(default_ppl)
        #   display(df_filter.loc[:, "Direct over Default"])
        #   flag = False
        df = pd.concat([df, df_filter], axis=0)
        print(f"Finished processing {base_model}, {quant}, {bit}")

  # Sort align with the tabular data
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                  'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                  '32-bit': 0, '8-bit': 1, '4-bit': 2, '3-bit': 3, '2-bit': 4,\
                  'Qwen/Qwen3-8B': 0, 'CohereLabs/aya-expanse-8b': 1, 'meta-llama/Llama-3.1-8B-Instruct': 2,}
  df = df.sort_values(by=['Quantization Technique', 'Bit Width', 'Base Model', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  pd.set_option('display.float_format','{:.3f}'.format)
  # pd.set_option('display.float_format','{:.1e}'.format)
  pd.set_option('display.max_rows', 500)
  if filter_result:
    display(df.loc[df['Calibration Language'] == filter_result, :])
  else:
    pd.set_option('display.float_format','{:.3f}'.format)
    display(df_unquantized_acc)
    pd.set_option('display.float_format','{:.3f}'.format)
    # pd.set_option('display.float_format','{:.1e}'.format)
    display(df)

  # for calib in calibration_languages:
  #   print(f"{compare_std=}: {calib}")
  #   display(df["Cross over Direct"][df["Calibration Language"] == calib])
  #   display(df["Cross over Direct"][df["Calibration Language"] == calib].describe())

  print("Coefficient of Variations")
  df_acc = df_acc.sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort)).reset_index(drop=True)
  df_acc_std = df_acc_std.sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort)).reset_index(drop=True)
  assert len(df_acc) == len(df_acc_std), "lenght of df_acc and df_acc_std is different"
  cov_df = df_acc.copy()
  cov_df.iloc[:, 4:] = (df_acc_std.iloc[:, 4:] / df_acc.iloc[:, 4:])
  display(cov_df)
  display(cov_df.describe())

  for eval_lang in eval_languages:
    calib_lang = reverse_calib_mapper.get(eval_lang, eval_lang)
    print(f"{eval_lang}: direct statistics")
    display(cov_df[cov_df["Calibration Language"] == calib_lang][eval_lang].describe())
    print(f"{eval_lang}: cross statistics")
    display(cov_df[cov_df["Calibration Language"] != calib_lang][eval_lang].describe())

  display(df["Cross over Direct"])
  display(df["Cross over Direct"].describe())
  if compare_std:
    display(df["Target"].describe())
    display(df["Mean Non Target"].describe())


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            {"config.base_model": "Qwen/Qwen3-8B"},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              {"config.base_model": "Qwen/Qwen3-8B"},
                            # {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_aggresive_target_not(eval_languages, runs, "", quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", hf_evaluate=True, gmean_aggregate=True)
  perplexity_aggresive_target_not(eval_languages, runs, "", quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", hf_evaluate=True, gmean_aggregate=True, compare_std=True)

/tmp/ipykernel_8238/3810827973.py:122: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filter = df_acc_std.loc[df_acc_std["Quantization Technique"] == quant][df_acc_std["Bit Width"] == bit][df_acc_std["Base Model"] == base_model] if compare_std else df_acc.loc[df_acc["Quantization Technique"] == quant][df_acc["Bit Width"] == bit][df_acc["Base Model"] == base_model]
/tmp/ipykernel_8238/3810827973.py:122: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filter = df_acc_std.loc[df_acc_std["Quantization Technique"] == quant][df_acc_std["Bit Width"] == bit][df_acc_std["Base Model"] == base_model] if compare_std else df_acc.loc[df_acc["Quantization Technique"] == quant][df_acc["Bit Width"] == bit][df_acc["Base Model"] == base_model]
/tmp/ipykernel_8238/3810827973.py:153: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or al

Finished processing Qwen/Qwen3-8B, gptq, 3-bit
Finished processing Qwen/Qwen3-8B, slimllm, 3-bit
Finished processing Qwen/Qwen3-8B, tacq, 3-bit


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.429,2.742,16.699,64.535,46.981


,Base Model,Quantization Technique,Bit Width,Calibration Language,Target,Mean Non Target,Direct over Unquantized,Cross over Unquantized,Direct over Default,Cross over Default,Direct over Cross,Cross over Direct,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,gptq,3-bit,English,40.525,42.644,121.228,127.568,0.000,-5.230,0.950,1.052,40.525,4.277,24.871,162.378,63.734
1,Qwen/Qwen3-8B,gptq,3-bit,Tamil,2.924,3.988,106.669,145.472,31.629,6.758,0.733,1.364,42.631,2.924,23.044,141.888,64.155
2,Qwen/Qwen3-8B,gptq,3-bit,Indonesian,19.075,23.721,114.226,142.052,23.305,4.622,0.804,1.244,42.928,3.789,19.075,136.644,62.711
3,Qwen/Qwen3-8B,gptq,3-bit,Swahili,79.112,150.236,122.589,232.799,51.279,7.477,0.527,1.899,44.148,3.725,22.242,79.112,65.088
4,Qwen/Qwen3-8B,gptq,3-bit,Chinese,55.975,63.917,119.144,136.048,12.174,-0.287,0.876,1.142,40.932,4.191,24.839,161.822,55.975
0,Qwen/Qwen3-8B,slimllm,3-bit,English,38.188,44.063,114.239,131.811,0.000,-15.382,0.867,1.154,38.188,4.949,26.285,164.319,64.713
1,Qwen/Qwen3-8B,slimllm,3-bit,Tamil,2.907,4.329,106.049,157.890,41.252,12.534,0.672,1.489,46.081,2.907,23.784,145.458,63.479
2,Qwen/Qwen3-8B,slimllm,3-bit,Indonesian,18.984,25.019,113.683,149.823,27.777,4.817,0.759,1.318,41.845,4.010,18.984,137.723,64.820
3,Qwen/Qwen3-8B,slimllm,3-bit,Swahili,74.108,155.937,114.834,241.633,54.900,5.101,0.475,2.104,44.450,3.972,22.584,74.108,68.374
4,Qwen/Qwen3-8B,slimllm,3-bit,Chinese,55.553,65.321,118.246,139.038,14.155,-0.940,0.850,1.176,43.979,4.455,27.751,179.624,55.553


Coefficient of Variations


...

,cmn_Hans
count,12.000
mean,0.013
std,0.005
min,0.006
25%,0.008
50%,0.013
75%,0.015
max,0.024


,Cross over Direct
0,1.463
1,6.783
2,3.484
3,3.557
4,1.114
0,3.559
1,12.168
2,14.326
3,8.210
4,1.219


,Cross over Direct
count,15.000
mean,4.894
std,4.053
min,0.903
25%,2.017
50%,3.557
75%,6.512
max,14.326


,Target
count,15.000
mean,0.328
std,0.359
min,0.004
25%,0.035
50%,0.175
75%,0.592
max,1.137


,Mean Non Target
count,15.000
mean,1.143
std,1.697
min,0.026
25%,0.118
50%,0.459
75%,0.977
max,5.831


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": "Qwen/Qwen3-8B"},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": "Qwen/Qwen3-8B"},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_aggresive_target_not(eval_languages, runs, "", quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", hf_evaluate=True, gmean_aggregate=True)
  perplexity_aggresive_target_not(eval_languages, runs, "", quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", hf_evaluate=True, gmean_aggregate=True, compare_std=True)

Finished processing CohereLabs/aya-expanse-8b, gptq, 2-bit
Finished processing Qwen/Qwen3-8B, gptq, 2-bit
Finished processing CohereLabs/aya-expanse-8b, slimllm, 2-bit
Finished processing Qwen/Qwen3-8B, slimllm, 2-bit
Finished processing CohereLabs/aya-expanse-8b, tacq, 2-bit
Finished processing Qwen/Qwen3-8B, tacq, 2-bit


/tmp/ipykernel_8238/3810827973.py:122: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filter = df_acc_std.loc[df_acc_std["Quantization Technique"] == quant][df_acc_std["Bit Width"] == bit][df_acc_std["Base Model"] == base_model] if compare_std else df_acc.loc[df_acc["Quantization Technique"] == quant][df_acc["Bit Width"] == bit][df_acc["Base Model"] == base_model]
/tmp/ipykernel_8238/3810827973.py:122: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filter = df_acc_std.loc[df_acc_std["Quantization Technique"] == quant][df_acc_std["Bit Width"] == bit][df_acc_std["Base Model"] == base_model] if compare_std else df_acc.loc[df_acc["Quantization Technique"] == quant][df_acc["Bit Width"] == bit][df_acc["Base Model"] == base_model]
/tmp/ipykernel_8238/3810827973.py:153: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or al

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.429,2.742,16.699,64.535,46.981
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.755,3.798,92.363,347.245,112.084


,Base Model,Quantization Technique,Bit Width,Calibration Language,Target,Mean Non Target,Direct over Unquantized,Cross over Unquantized,Direct over Default,Cross over Default,Direct over Cross,Cross over Direct,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,gptq,2-bit,English,12827.902,38022.199,38373.920,113741.192,0.000,-196.402,0.337,2.964,12827.902,45940.807,39236.778,40398.302,39491.466
1,Qwen/Qwen3-8B,gptq,2-bit,Tamil,7.469,33666.480,272.430,1227984.338,99.984,26.718,0.000,4507.526,43858.254,7.469,46203.200,46825.504,34007.537
2,Qwen/Qwen3-8B,gptq,2-bit,Indonesian,1683.806,41260.315,10083.272,247082.435,95.709,-5.157,0.041,24.504,35069.945,32266.297,1683.806,34796.007,42150.094
3,Qwen/Qwen3-8B,gptq,2-bit,Swahili,1466.964,40627.737,2273.142,62954.924,96.369,-0.568,0.036,27.695,44051.236,37346.013,41954.948,1466.964,25431.656
4,Qwen/Qwen3-8B,gptq,2-bit,Chinese,17781.025,34638.805,37847.424,73729.696,54.975,12.288,0.513,1.948,30846.395,23205.924,38104.965,41391.916,17781.025
0,CohereLabs/aya-expanse-8b,gptq,2-bit,English,241.836,754.221,449.887,1403.073,0.000,-211.873,0.321,3.119,241.836,22325.499,6090.955,22225.968,19168.145
1,CohereLabs/aya-expanse-8b,gptq,2-bit,Tamil,6.436,4939.166,169.450,130035.052,99.971,77.877,0.001,767.394,745.679,6.436,6473.672,21938.559,9278.961
2,CohereLabs/aya-expanse-8b,gptq,2-bit,Indonesian,545.812,5693.192,590.940,6163.915,91.039,6.530,0.096,10.431,522.582,4200.198,545.812,14221.137,9950.510
3,CohereLabs/aya-expanse-8b,gptq,2-bit,Swahili,1591.608,19500.949,458.353,5615.900,92.839,12.261,0.082,12.252,824.976,1501.481,4321.038,1591.608,14767.553
4,CohereLabs/aya-expanse-8b,gptq,2-bit,Chinese,1029.857,12714.765,918.824,11343.938,94.627,33.667,0.081,12.346,1006.575,4226.914,6165.939,20855.464,1029.857


Coefficient of Variations


...

,cmn_Hans
count,23.000
mean,0.184
std,0.228
min,0.008
25%,0.037
50%,0.081
75%,0.293
max,0.895


,Cross over Direct
0,5.126
1,5859.364
2,5.249
3,8.937
4,1.412
0,8.736
1,5994.001
2,17.328
3,49.370
4,63.205


,Cross over Direct
count,29.000
mean,5251.453
std,18128.686
min,1.412
25%,8.736
50%,20.843
75%,84.464
max,86417.678


,Target
count,29.000
mean,97.643
std,291.155
min,0.010
25%,0.387
50%,8.992
75%,37.524
max,1521.219


,Mean Non Target
count,29.000
mean,30086.603
std,129291.113
min,0.559
25%,8.075
50%,1066.372
75%,2148.070
max,687490.834


##### Stats pool model

In [ ]:
def perplexity_aggresive_target_not_pool_model(eval_languages, runs, base_model, quantization_technique, bit_width, eval_dataset, custom_title="", filter_result='', hf_evaluate=False, compare_std=False, gmean_aggregate=False):
  language_sort = {
      # Flores Plus
      "eng_Latn": 0, "fra_Latn": 1, "hin_Deva": 2, "npi_Deva": 3, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "cmn_Hans": 12, "jpn_Jpan": 13, "kor_Hang": 14, "arb_Arab" : 15,\
      # Wikipedia
      "en": 0, "fr": 1, "hi": 2, "ne": 3, "ta": 6, "te": 7, "id": 8, "ms": 9, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
  }
  flores_plus_add = ["afr_Latn", "dan_Latn", "deu_Latn", "eng_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 0 + index / 100
  flores_plus_add = ["kan_Knda", "mal_Mlym", "tam_Taml", "tel_Telu"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 6 + index / 100
  #flores_plus_add = ["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "ind_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"]
  flores_plus_add = ["ind_Latn", "zsm_Latn", "bjn_Latn", "min_Latn", "ace_Latn", "ban_Latn", "bug_Latn", "jav_Latn", "sun_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "fil_Latn", "war_Latn"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 8 + index / 100
  flores_plus_add = ["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "swh_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 10 + index / 100
  flores_plus_add = ["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant", "cmn_Hans"]
  for index, lang in enumerate(flores_plus_add):
    language_sort[lang] = 12 + index / 100
  eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))

  calib_mapper = {
      "English": "eng_Latn" if eval_dataset.lower() == "floresplus" else "en",
      "Indonesian": "ind_Latn" if eval_dataset.lower() == "floresplus" else "id",
      "Tamil": "tam_Taml" if eval_dataset.lower() == "floresplus" else "ta",
      "Swahili": "swh_Latn" if eval_dataset.lower() == "floresplus" else "sw",
      "Chinese": "cmn_Hans" if eval_dataset.lower() == "floresplus" else "zh",
  }

  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language"]
  columns.extend(eval_languages)
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 '32-bit': 0, '2-bit': 1, '3-bit': 2, '4-bit': 3, '8-bit': 4}

  data_acc = []
  data_nancount = []

  unquantized_data_acc = []


  # Iterate over the runs and print details
  for run in runs:
      config = run.config
      summary = run.summary

      if config["calibration_language"] == "WuuChinese":
        continue

      run_data_acc = []
      run_data_nancount = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      run_data_nancount.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config["calibration_language"]])
      if not hf_evaluate:
        for lang in eval_languages:
          if lang in ["zh", "Chinese", "cmn_Hans", "ja", "Japanese", "jpn_Jpan"]:
            acc = run.summary.get(f"{eval_dataset.lower()}_byte_acc_{lang}", None)
            nancount = run.summary.get(f"{eval_dataset.lower()}_count_nan_count_{lang}", 0)
            if acc:
              # acc = math.pow(acc, 1/3)
              pass
            run_data_acc.append(acc)
            run_data_nancount.append(nancount)
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_word_acc_{lang}", None))
            run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nan_count_{lang}", 0))
      elif hf_evaluate:
        for lang in eval_languages:
          if gmean_aggregate:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_gmean_perplexity_{lang}", None))
          else:
            run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_token_perplexity_{lang}", None))
          run_data_nancount.append(run.summary.get(f"{eval_dataset.lower()}_nancount_{lang}", 0))

      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
        data_nancount.append(run_data_nancount)
      else:
        data_acc.append(run_data_acc)
        data_nancount.append(run_data_nancount)


  import numpy as np
  df_unquantized_acc = pd.DataFrame(unquantized_data_acc, columns=columns)
  df_unquantized_acc_chief_std = df_unquantized_acc.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std")

  df_acc = pd.DataFrame(data_acc, columns=columns).sort_values(by=['Quantization Technique', 'Calibration Language', 'Bit Width'], key=lambda x: x.map(custom_sort))
  df_acc_std = df_acc.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_acc = df_acc.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  df_acc[df_acc.columns[4:]] = df_acc[df_acc.columns[4:]].astype(float)
  df_unquantized_acc[df_unquantized_acc.columns[4:]] = df_unquantized_acc[df_unquantized_acc.columns[4:]].astype(float)

  pd.set_option('display.float_format','{:.1f}'.format)
  # display(df_unquantized_acc)

  # # Perplexity related, change order of PPL
  # df_acc.iloc[:, 5:] = df_acc.iloc[:, 5:].apply(np.log10)
  # df_unquantized_acc.iloc[:, 5:] = df_unquantized_acc.iloc[:, 5:].apply(np.log10)

  calibration_languages = df_acc["Calibration Language"].unique()
  quants = df_acc["Quantization Technique"].unique()
  base_models = df_acc["Base Model"].unique()
  bits = df_acc["Bit Width"].unique()

  flag = True
  df = pd.DataFrame(data=[], columns=["Base Model", "Quantization Technique", "Bit Width", "Calibration Language", "Target", "Mean Non Target",
                                      "Direct over Unquantized", "Cross over Unquantized", "Direct over Default", "Cross over Default", "Direct over Cross", "Cross over Direct"])
  for quant in quants:
    for bit in bits:
      for base_model in base_models:
        df_filter_uq = df_unquantized_acc[df_unquantized_acc["Base Model"] == base_model]
        df_filter = df_acc_std.loc[df_acc_std["Quantization Technique"] == quant][df_acc_std["Bit Width"] == bit][df_acc_std["Base Model"] == base_model] if compare_std else df_acc.loc[df_acc["Quantization Technique"] == quant][df_acc["Bit Width"] == bit][df_acc["Base Model"] == base_model]
        mean_ppl = df_filter.iloc[:, 4:].prod(axis=0)
        default_ppl = df_filter[df_filter["Calibration Language"] == "English"].iloc[0, 4:]
        count_ppl = df_filter.iloc[:, 4:].count(axis=0)
        df_filter = df_filter.reset_index(drop=True)
        if (quant, base_model) == ("slimllm", "CohereLabs/aya-expanse-8b"):
          mean_ppl = mean_ppl.drop(calib_mapper["Tamil"], axis=0)
          default_ppl = default_ppl.drop(calib_mapper["Tamil"], axis=0)
          count_ppl = count_ppl.drop(calib_mapper["Tamil"], axis=0)
          df_filter_uq = df_filter_uq.drop(calib_mapper["Tamil"], axis=1)

        df_filter.loc[:, "Target"] = pd.Series([df_filter.loc[i, calib_mapper[calib]] for i, calib in enumerate(df_filter["Calibration Language"].unique())])
        df_filter.loc[:, "Mean Non Target"] = (mean_ppl.values / df_filter["Target"].values) ** (1/(count_ppl.values - 1))
        # df_filter.loc[:, "Power Ratio"] = (10 ** df_filter["Mean Non Target"].values / 10 ** df_filter["Target"].values)

        # [0, x:] index is transforming from 2D (df) to 1D array (series)
        # This will broke if Unquantized have more than 1 instance
        df_filter.loc[:, "Direct over Unquantized"] = (df_filter["Target"].values / df_filter_uq.iloc[0, 4:].values) * 100
        df_filter.loc[:, "Cross over Unquantized"] = (df_filter["Mean Non Target"].values / df_filter_uq.iloc[0, 4:].values) * 100
        df_filter.loc[:, "Direct over Default"] = ((default_ppl.values - df_filter["Target"].values) / default_ppl.values) *100
        df_filter.loc[:, "Cross over Default"] = ((default_ppl.values - df_filter["Mean Non Target"].values) / default_ppl.values) * 100
        # df_filter.loc[:, "Direct over Cross"] = (df_filter["Target"].values / df_filter["Mean Non Target"].values) * 100
        df_filter.loc[:, "Direct over Cross"] = (df_filter["Target"].values / df_filter["Mean Non Target"].values)
        df_filter.loc[:, "Cross over Direct"] = (df_filter["Mean Non Target"].values / df_filter["Target"].values)
        # df_filter.drop(df_filter.columns[5:].drop(["Target", "Mean Non Target"]), axis=1)
        # if flag:
        #   print(quant, bit, base_model)
        #   display(df_filter["Target"])
        #   display(default_ppl)
        #   display(df_filter.loc[:, "Direct over Default"])
        #   flag = False
        df = pd.concat([df, df_filter], axis=0)
        print(f"Finished processing {base_model}, {quant}, {bit}")

  # Sort align with the tabular data
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                  'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                  '32-bit': 0, '8-bit': 1, '4-bit': 2, '3-bit': 3, '2-bit': 4,\
                  'Qwen/Qwen3-8B': 0, 'CohereLabs/aya-expanse-8b': 1, 'meta-llama/Llama-3.1-8B-Instruct': 2,}
  df = df.sort_values(by=['Quantization Technique', 'Bit Width', 'Base Model', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  pd.set_option('display.float_format','{:.3f}'.format)
  # pd.set_option('display.float_format','{:.1e}'.format)
  pd.set_option('display.max_rows', 500)
  if filter_result:
    display(df.loc[df['Calibration Language'] == filter_result, :])
  else:
    pd.set_option('display.float_format','{:.3f}'.format)
    display(df_unquantized_acc)
    pd.set_option('display.float_format','{:.3f}'.format)
    # pd.set_option('display.float_format','{:.1e}'.format)
    display(df)

  # for calib in calibration_languages:
  #   print(f"{compare_std=}: {calib}")
  #   display(df["Cross over Direct"][df["Calibration Language"] == calib])
  #   display(df["Cross over Direct"][df["Calibration Language"] == calib].describe())

  print("Coefficient of Variations")
  cov_std = df_acc_std.drop("Base Model", axis=1).groupby(['Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  cov_acc = df_acc.drop("Base Model", axis=1).groupby(['Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  cov = (cov_std.iloc[:, 3:] / cov_acc.iloc[:, 3:]).rename(df_acc["Quantization Technique"] + '-' + df_acc["Calibration Language"])
  display(cov)
  display(cov.describe())

  display(df["Cross over Direct"])
  display(df["Cross over Direct"].describe())
  if compare_std:
    display(df["Target"].describe())
    display(df["Mean Non Target"].describe())


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Initialize the W&B API
api = Api()

# eval_languages = ["arb_Arab", "eng_Latn", "cmn_Hans", "hin_Deva", "fra_Latn", "jpn_Jpan", "zsm_Latn", "kor_Hang", "tam_Taml", "ind_Latn", "swh_Latn", "tel_Telu", "npi_Deva", "yor_Latn"]
quantization_technique = "ALL"
bit_width = 8
evaluation_dataset = "floresplus"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit_width in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"config.bit_width": f"{bit_width}-bit"},
                            {"config.num_calibration_samples": 128},
                            # {"config.evaluation_dataset": evaluation_dataset},
                            {"$or": [
                              {"config.evaluation_dataset": f"floresplus-evaluate"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-1000"},
                              {"config.evaluation_dataset": f"floresplus-evaluate-2000"},
                            ]},
                            # {"config.base_model": "Qwen/Qwen3-8B"},
                            {"config.enable_thinking": False},
                            {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                            {"$not": {"$and": [{"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}, {"config.quantization_technique": "slimllm"}, {"config.calibration_language": "Tamil"}]}}
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": evaluation_dataset},
                              {"config.evaluation_dataset": f"{evaluation_dataset}-evaluate"},
                              # {"config.base_model": "Qwen/Qwen3-8B"},
                            {"$not":  {"config.base_model": "meta-llama/Llama-3.1-8B-Instruct"}},
                            # {"$not":  {"config.base_model": "CohereLabs/aya-expanse-8b"}},
                              {"config.enable_thinking": False},
                              {"config.output_type": "perplexity"},
                            {"created_at": {"$gt": '2026-01-15'}},
                            {"created_at": {"$lt": '2026-07-01'}},
                          ]}
                      ]}
                  )

  eval_languages = ["eng_Latn", "cmn_Hans", "tam_Taml", "ind_Latn", "swh_Latn"]
  # eval_languages.extend(["afr_Latn", "dan_Latn", "deu_Latn", "fao_Latn", "isl_Latn", "lim_Latn", "ltz_Latn", "nld_Latn", "nno_Latn", "nob_Latn", "swe_Latn", "tpi_Latn", "ydd_Hebr"])
  # eval_languages.extend(["kan_Knda", "mal_Mlym", "tel_Telu"])
  # eval_languages.extend(["ace_Latn", "ban_Latn", "bjn_Latn", "bug_Latn", "ceb_Latn", "fij_Latn", "ilo_Latn", "jav_Latn", "min_Latn", "plt_Latn", "mri_Latn", "pag_Latn", "smo_Latn", "sun_Latn", "fil_Latn", "war_Latn", "zsm_Latn"])
  # eval_languages.extend(["bem_Latn", "cjk_Latn", "ibo_Latn", "kam_Latn", "kik_Latn", "kin_Latn", "kmb_Latn", "ktu_Latn", "lin_Latn", "lua_Latn", "lug_Latn", "nso_Latn", "nya_Latn", "run_Latn", "sna_Latn", "sot_Latn", "ssw_Latn", "tsn_Latn", "tso_Latn", "tum_Latn", "umb_Latn", "xho_Latn", "yor_Latn", "zul_Latn"])
  # eval_languages.extend(["bod_Tibt", "dzo_Tibt", "kac_Latn", "lus_Latn", "mni_Beng", "mya_Mymr", "yue_Hant"])

  # perplexity_heatmap_deliver_vertical(eval_languages, runs, ["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", display_mean=False, compile=False, show_subline=True, hf_evaluate=True, zscore_threshold=1/2, gmean_aggregate=False)
  perplexity_aggresive_target_not_pool_model(eval_languages, runs, "", quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", hf_evaluate=True, gmean_aggregate=True)
  # perplexity_aggresive_target_not(eval_languages, runs, "", quantization_technique, bit_width, evaluation_dataset, custom_title="All-PPL", hf_evaluate=True, gmean_aggregate=True, compare_std=True)

Finished processing CohereLabs/aya-expanse-8b, gptq, 2-bit
Finished processing Qwen/Qwen3-8B, gptq, 2-bit
Finished processing CohereLabs/aya-expanse-8b, slimllm, 2-bit
Finished processing Qwen/Qwen3-8B, slimllm, 2-bit
Finished processing CohereLabs/aya-expanse-8b, tacq, 2-bit
Finished processing Qwen/Qwen3-8B, tacq, 2-bit


/tmp/ipykernel_12153/1680869954.py:121: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_12153/1680869954.py:121: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_12153/1680869954.py:152: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/tmp/ipykernel_12153/1680869954.py:121: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_12153/1680869954.py:121: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_12153/1680869954.py:121: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_12153/1680869954.py:121: UserWarning:

Boolean Series key w

,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,Unquantized,32-bit,Unquantized,33.429,2.742,16.699,64.535,46.981
1,CohereLabs/aya-expanse-8b,Unquantized,32-bit,Unquantized,53.755,3.798,92.363,347.245,112.084


,Base Model,Quantization Technique,Bit Width,Calibration Language,Target,Mean Non Target,Direct over Unquantized,Cross over Unquantized,Direct over Default,Cross over Default,Direct over Cross,Cross over Direct,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
0,Qwen/Qwen3-8B,gptq,2-bit,English,12827.902,38022.199,38373.920,113741.192,0.000,-196.402,0.337,2.964,12827.902,45940.807,39236.778,40398.302,39491.466
1,Qwen/Qwen3-8B,gptq,2-bit,Tamil,7.469,33666.480,272.430,1227984.338,99.984,26.718,0.000,4507.526,43858.254,7.469,46203.200,46825.504,34007.537
2,Qwen/Qwen3-8B,gptq,2-bit,Indonesian,1683.806,41260.315,10083.272,247082.435,95.709,-5.157,0.041,24.504,35069.945,32266.297,1683.806,34796.007,42150.094
3,Qwen/Qwen3-8B,gptq,2-bit,Swahili,1466.964,40627.737,2273.142,62954.924,96.369,-0.568,0.036,27.695,44051.236,37346.013,41954.948,1466.964,25431.656
4,Qwen/Qwen3-8B,gptq,2-bit,Chinese,17781.025,34638.805,37847.424,73729.696,54.975,12.288,0.513,1.948,30846.395,23205.924,38104.965,41391.916,17781.025
0,CohereLabs/aya-expanse-8b,gptq,2-bit,English,241.836,754.221,449.887,1403.073,0.000,-211.873,0.321,3.119,241.836,22325.499,6090.955,22225.968,19168.145
1,CohereLabs/aya-expanse-8b,gptq,2-bit,Tamil,6.436,4939.166,169.450,130035.052,99.971,77.877,0.001,767.394,745.679,6.436,6473.672,21938.559,9278.961
2,CohereLabs/aya-expanse-8b,gptq,2-bit,Indonesian,545.812,5693.192,590.940,6163.915,91.039,6.530,0.096,10.431,522.582,4200.198,545.812,14221.137,9950.510
3,CohereLabs/aya-expanse-8b,gptq,2-bit,Swahili,1591.608,19500.949,458.353,5615.900,92.839,12.261,0.082,12.252,824.976,1501.481,4321.038,1591.608,14767.553
4,CohereLabs/aya-expanse-8b,gptq,2-bit,Chinese,1029.857,12714.765,918.824,11343.938,94.627,33.667,0.081,12.346,1006.575,4226.914,6165.939,20855.464,1029.857


Coefficient of Variations


,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
gptq-English,0.021,0.152,0.031,0.045,0.100
gptq-Tamil,0.072,0.047,0.120,0.030,0.322
gptq-Indonesian,0.042,0.124,0.224,0.089,0.051
gptq-Swahili,0.038,0.054,0.059,0.050,0.035
gptq-Chinese,0.018,0.107,0.121,0.137,0.082
slimllm-English,0.083,0.687,0.331,0.182,0.181
tacq-Chinese,0.375,0.024,0.190,0.206,0.383
slimllm-Indonesian,0.441,0.724,0.211,0.287,0.271
slimllm-Swahili,0.398,1.358,0.270,0.199,0.741
slimllm-Chinese,0.560,0.902,0.103,0.463,0.093


,eng_Latn,tam_Taml,ind_Latn,swh_Latn,cmn_Hans
count,15.000,15.000,15.000,15.000,15.000
mean,0.140,0.296,0.123,0.145,0.159
std,0.194,0.415,0.100,0.114,0.200
min,0.005,0.003,0.013,0.026,0.006
25%,0.014,0.050,0.043,0.070,0.033
50%,0.038,0.099,0.103,0.123,0.082
75%,0.229,0.420,0.200,0.190,0.226
max,0.560,1.358,0.331,0.463,0.741


,Cross over Direct
0,2.964
1,4507.526
2,24.504
3,27.695
4,1.948
0,3.119
1,767.394
2,10.431
3,12.252
4,12.346


,Cross over Direct
count,29.000
mean,614.278
std,1788.508
min,1.289
25%,3.036
50%,10.431
75%,33.726
max,8386.566
